In [ ]:
library(Seurat)

In [ ]:
own <- readRDS("/mnt/90-connectome/Personal/XQ/a-nc-raphe/step5_data_save_1019/classFiltSeurat_filt_allSubtypes.rds")

# 1. Basic structure
own
head(rownames(own), 20)        # gene names
head(colnames(own), 5)         # cell barcodes
dim(own)                       # genes x cells

# 2. Metadata columns
colnames(own@meta.data)

# 3. Broad/fine cell-type distribution
sort(table(own$cell_type_fine), decreasing = TRUE)

# 4. Find 5-HT-related clusters
grep("5-?HT|serot|raphe", unique(own$cell_type_fine),
     value = TRUE, ignore.case = TRUE)

# 5. Distribution of predicted broad classes (used for coarse annotation when merging)
if ("maxPredClass" %in% colnames(own@meta.data)) {
  sort(table(own$maxPredClass), decreasing = TRUE)
} else {
  cat("no maxPredClass column\n")
}

In [ ]:
# Cross-species

In [ ]:
library(Seurat)

all_seu <- readRDS("/mnt/90-connectome/Personal/XQ/a-nc-raphe/step5_data_save_1019/classFiltSeurat_filt_allSubtypes.rds")

# ---- 1. Object overview: assays, cell counts, gene counts, reductions ----
all_seu

# ---- 2. All metadata column names ----
colnames(all_seu@meta.data)

# ---- 3. Inspect each column (first 10 rows) ----
for (cc in colnames(all_seu@meta.data)) {
  cat("\n========== ", cc, " ==========\n")
  print(head(all_seu@meta.data[[cc]], 10))
}

# ---- 4. Value distributions of key columns ----
# Broad classes
table(all_seu$cell_class)

# Subtypes (may be long; just check levels and counts)
length(levels(factor(all_seu$cell_type_new)))
sort(table(all_seu$cell_type_new), decreasing = TRUE)

# donor
table(all_seu$donor)

# ---- 5. Available assays / layers ----
Assays(all_seu)
Layers(all_seu[["RNA"]])
Layers(all_seu[["SCT"]])

# ---- 6. Available reductions and their dimensions ----
Reductions(all_seu)
sapply(Reductions(all_seu), function(r) dim(Embeddings(all_seu, r)))

# ---- 7. Clustering results ----
grep("snn_res", colnames(all_seu@meta.data), value = TRUE)
table(all_seu$seurat_clusters)

# ---- 8. What the current Idents are ----
levels(Idents(all_seu))

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/step5_data_save_907/"
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1028/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## =========================================================================
## Part 1: load annotated data, keep only 5-HT cells
## =========================================================================

own <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/step5_data_save_907/ht_recluster/ht_5ht_nFeature1500_reclustered_named.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT subtype composition ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## Part 2: cross-species integration (macaque x mouse, CCA, full dataset, no downsampling)
## =========================================================================

PAR <- list(
  variable_genes = 3000,
  npcs           = 100,
  cca_dims       = 1:80,
  nb_dims        = 1:30,
  resolution     = 0.1,
  min_cells      = 100,
  k_anchor       = 5
)

## ---- 2.1 Load data + clean ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 One-to-one ortholog conversion ----
test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 Batch labels (keep all cells, no downsampling) ----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

cat("Full dataset (no downsampling):\n")
print(table(macaqueHomoSeurat$batch))
print(table(mouseHomoSeurat$batch))
cat("macaque total:", ncol(macaqueHomoSeurat),
    " mouse total:", ncol(mouseHomoSeurat),
    " combined:", ncol(macaqueHomoSeurat) + ncol(mouseHomoSeurat), "\n")

## ---- 2.4 split + merge ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 Clustering + UMAP ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 Visualization ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE, pt.size = 1.0) +
  coord_fixed() + ggtitle("5-HT cross-species (all cells, CCA dims80)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols, pt.size = 1.0) +
  coord_fixed() + ggtitle("batch check (all cells)")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_batch.pdf"), p2, width = 6.5, height = 5)

mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("cells missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species_allcells.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE, pt.size = 1.2) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (all cells)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_subtype_split.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE,
              pt.size = 0.6) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_subtype_all.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap heatmap ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap matrix:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(save_dir, "overlap_cca_allcells_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(save_dir, "overlap_cca_allcells_mac5HT.csv"))
cat("Overlap heatmap saved\n")

# ## ---- 2.9 Save integrated object (the full version is large; be sure to save) ----
# saveRDS(classFiltSeurat, paste0(save_dir, "cross_species_5HT_allcells_cca.rds"))
# cat("Integrated object saved\n")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "/mnt/90-connectome/Personal/XQ/a-nc-raphe/step5_data_save_1019/"
save_dir <- "/mnt/90-connectome/Personal/XQ/a-nc-raphe/cross_species/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## =========================================================================
## Part 1: load annotated data, keep only 5-HT cells
## =========================================================================

own <- readRDS("/mnt/90-connectome/Personal/XQ/a-nc-raphe/step5_data_save_1019/classFiltSeurat_filt_allSubtypes.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT subtype composition ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## Part 2: cross-species integration (macaque x mouse, CCA, batches downsampled)
## =========================================================================

PAR <- list(
  variable_genes = 3000,
  npcs           = 100,
  cca_dims       = 1:80,
  nb_dims        = 1:30,
  resolution     = 0.1,
  min_cells      = 100,
  k_anchor       = 5
)

## ---- 2.1 Load data + clean ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 One-to-one ortholog conversion ----
test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 Merge small donors + downsample each batch ----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("Downsample each batch to ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 split + merge ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 Clustering + UMAP ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 Visualization (pt.size increased further) ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE,
              pt.size = 1.0) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species1.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols,
              pt.size = 1.0) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch1.pdf"), p2, width = 6.5, height = 5)

mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("cells missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE,
              pt.size = 1.2) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend1.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE,
              pt.size = 0.6) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend1.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap heatmap ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap matrix:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(save_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(save_dir, "overlap_cca_dims50_mac5HT1.csv"))
cat("Overlap heatmap saved\n")

In [ ]:
## ---- 2.7 Visualization (pt.size increased further) ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE,
              pt.size = 1.5
            ) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)


In [ ]:
# suppressMessages({
#   library(tidyverse)
#   library(Seurat)
#   library(patchwork)
#   library(Matrix)
# })

# set.seed(42)
# out_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/"
# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"
# dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ## =========================================================================
# ## Part 1: load annotated data, keep only 5-HT cells (no reclustering; use cell_type_new directly)
# ## =========================================================================

# own <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/step5_data_save_907/all_cells_recluster/all_cells_reclustered.rds")
# own[["RNA"]] <- JoinLayers(own[["RNA"]])

# mcq <- subset(own, subset = cell_type_new %in%
#                 grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
# mcq$renew_type <- as.character(mcq$cell_type_new)

# cat("--- macaque 5-HT subtype composition ---\n")
# print(table(mcq$renew_type, mcq$donor))

# ## =========================================================================
# ## Part 2: cross-species integration (macaque x mouse, CCA dims 1:50, batches downsampled)
# ## =========================================================================

# PAR <- list(
#   variable_genes = 8000,
#   npcs           = 100,
#   cca_dims       = 1:80,
#   nb_dims        = 1:30,
#   resolution     = 0.1,
#   min_cells      = 100,
#   k_anchor       = 5
# )

# ## ---- 2.1 Load data + clean ----
# macaque <- mcq
# DefaultAssay(macaque) <- "RNA"
# macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
# macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
# macaque$donor <- as.character(macaque$donor)
# macaque$maxPredClass <- "5-HT"
# Idents(macaque) <- "renew_type"

# mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
# mouse$maxPredClass <- "5-HT"
# mouse$donor <- mouse$group
# Idents(mouse) <- "renew_type"

# ## ---- 2.2 One-to-one ortholog conversion ----
# test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
#                    sep = "\t", header = TRUE)
# comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
# colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
# comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
# comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
# comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

# prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
#                               keepMetaCol, commonMetaCol,
#                               speciesPrefix = "macaque_", clusterCol = "renew_type") {
#   homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
#   rownames(homoCountMx) <- homoGeneName
#   colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
#   homoSeurat <- CreateSeuratObject(counts = homoCountMx)
#   tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
#                                 colnames(homoSeurat)), keepMetaCol]
#   tmpDf$species <- sub("(_|-)", "", speciesPrefix)
#   tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
#   commonMetaCol <- c(commonMetaCol, "species", "clusterName")
#   colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
#                             colnames(tmpDf),
#                             paste0(speciesPrefix, colnames(tmpDf)))
#   rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
#   homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
#   return(homoSeurat)
# }

# comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
# comGeneTb <- subset(comGeneTb,
#   macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
#   mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
# message("One-to-one orthologs: ", nrow(comGeneTb))

# macaqueHomoSeurat <- prepareHomoSeurat(
#   macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
#   keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
#                   "renew_type", "donor"),
#   commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
#   speciesPrefix = "macaque_"
# )
# mouseHomoSeurat <- prepareHomoSeurat(
#   mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
#   keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
#                   "renew_type", "group", "donor"),
#   commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
#   speciesPrefix = "mouse_"
# )
# mouseHomoSeurat$species   <- "mouse"
# macaqueHomoSeurat$species <- "macaque"

# ## ---- 2.3 Merge small donors + downsample each batch ----
# own_donors <- c("Mq246", "Mq277")
# macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
#                                   macaqueHomoSeurat$donor, "NHP_published")
# mouseHomoSeurat$batch <- mouseHomoSeurat$donor

# target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
#                           table(mouseHomoSeurat$batch)))
# message("Downsample each batch to ", target_per_batch)

# downsample_by_batch <- function(obj, target_per_batch) {
#   ct <- obj$clusterName; bt <- obj$batch
#   cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
#     n_target <- min(length(idx), target_per_batch)
#     tab <- table(ct[idx])
#     n_per <- round(tab / sum(tab) * n_target)
#     n_per <- pmin(n_per, tab)
#     n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
#     unlist(lapply(names(n_per), function(x) {
#       cells_x <- colnames(obj)[idx][ct[idx] == x]
#       sample(cells_x, min(n_per[x], length(cells_x)))
#     }))
#   }))
#   sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
# }
# macaqueHomoSeurat <- subset(macaqueHomoSeurat,
#                             cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
# mouseHomoSeurat   <- subset(mouseHomoSeurat,
#                             cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
# print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

# ## ---- 2.4 split + merge ----
# mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
# macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
# classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
# print(sapply(Layers(classFiltSeurat[["RNA"]]),
#              function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

# ## ---- 2.5 SCT + PCA + CCA ----
# classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
#                                ncells = ncol(classFiltSeurat[["RNA"]]),
#                                variable.features.n = PAR$variable_genes,
#                                method = "glmGamPoi")
# classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
# classFiltSeurat <- IntegrateLayers(classFiltSeurat,
#                                    method = CCAIntegration,
#                                    orig.reduction = "pca",
#                                    new.reduction = "integrated.cca",
#                                    normalization.method = "SCT",
#                                    dims = PAR$cca_dims, verbose = TRUE)

# ## ---- 2.6 Clustering + UMAP ----
# classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
#                                  dims = PAR$nb_dims)
# classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
# classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
#                            dims = PAR$nb_dims, reduction.name = "umap")

# ## ---- 2.7 Visualization (pt.size increased) ----
# library(ggplot2)
# colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
# batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
#                "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
#                "NHP_published" = "#F5B971")

# p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
#               cols = colorLeg, raster = FALSE,
#               pt.size = 0.6) +
#   coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
# print(p1)
# ggsave(paste0(save_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

# p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
#               cols = batchCols,
#               pt.size = 0.6) +
#   coord_fixed() + ggtitle("batch check")
# print(p2)
# ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
# mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
# mac_stripped <- sub("^macaque_", "", mac_cells_cf)

# classFiltSeurat$subtype_recl <- NA
# classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
# classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
#   sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

# n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
# cat("cells missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")
# classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
# classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

# print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
# write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
#           paste0(save_dir, "subtype_recl_vs_species.csv"))

# all_types <- levels(classFiltSeurat$subtype_recl)
# mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
# mus_types_lbl <- setdiff(all_types, mac_types_lbl)
# subtype_cols <- c(
#   setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
#   setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
# )
# legend_theme <- theme(legend.position = "right",
#                       legend.title = element_text(size = 10),
#                       legend.text = element_text(size = 8),
#                       legend.key.size = unit(0.4, "cm"))
# legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
#                                             override.aes = list(size = 3)))

# p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
#               split.by = "species", label = TRUE, repel = TRUE,
#               label.size = 3.5, ncol = 2, raster = FALSE,
#               pt.size = 0.8) +
#   coord_fixed() + scale_color_manual(values = subtype_cols) +
#   ggtitle("subtypes split (macaque = annotated 5-HT)") +
#   legend_theme + legend_guide
# print(p3)
# ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend.pdf"),
#        p3, width = 15, height = 6.5)

# p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
#               label = TRUE, repel = TRUE, label.size = 3, raster = FALSE,
#               pt.size = 0.4) +
#   coord_fixed() + scale_color_manual(values = subtype_cols) +
#   legend_theme + legend_guide
# print(p4)
# ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend.pdf"),
#        p4, width = 10, height = 6.5)

# ## ---- 2.8 Overlap heatmap ----
# suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

# compare_cl <- function(cl, ref.cl) {
#   conf1 <- table(cl, ref.cl)
#   conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
#   cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
#   cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
#                            dimnames = list(rownames(conf1), rownames(conf1)))
#   return(list(conf = conf1, cocl = cl.prop.cocl.m))
# }

# cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
# cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
#                             grepl("^5-HT",    colnames(cl.conf$cocl))]
# cat("Overlap matrix:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

# row_max <- apply(cocl.subset, 1, max)
# bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
# left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
#   row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
#   border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
# bar_legend <- Legend(title = "Type",
#                      at = c("macaque-specific", "macaque/mouse-conserved"),
#                      legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
#                      border = "black", labels_gp = gpar(fontsize = 10))

# ht <- Heatmap(cocl.subset, name = "Overlap score",
#               col = colorRampPalette(c("white", "grey70", "black"))(100),
#               cluster_rows = FALSE, cluster_columns = FALSE,
#               left_annotation = left_anno, row_names_side = "right",
#               row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
#               width = unit(6, "cm"), height = unit(8, "cm"))

# png(paste0(out_dir, "overlap_cca_dims50_mac5HT.png"),
#     width = 8.5, height = 8, units = "in", res = 300)
# draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
# decorate_annotation("Max Overlap\nscore", {
#   n <- length(row_max)
#   for (j in seq_len(n))
#     grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
#               y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
# })
# decorate_annotation("Max Overlap\nscore", {
#   grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
# })
# dev.off()

# write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_mac5HT.csv"))
# cat("Overlap heatmap saved\n")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/"
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)
## =========================================================================
## Part 1: load annotated data, keep only 5-HT cells (no reclustering; use cell_type_new directly)
## =========================================================================

own <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/step5_data_save_907/all_cells_recluster/all_cells_reclustered.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

# keep only 5-HT cells, reuse existing annotations
mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT subtype composition ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## Part 2: cross-species integration (macaque x mouse, CCA dims 1:50, batches downsampled)
## =========================================================================

PAR <- list(
  variable_genes = 8000,    # 8000 -> 10000: approaching the one-to-one ortholog limit
  npcs           = 100,
  cca_dims       = 1:80,     # 60 -> 80: relax the anchor-related space further
  nb_dims        = 1:30,     # 50 -> 40: tighten graph dims, filter out CCA tail noise
  resolution     = 0.1,      # unchanged (again: res does not affect integration, only cluster granularity)
  min_cells      = 100,
  k_anchor       = 5        # new: anchor k value; default 5 -> 10
)

## ---- 2.1 Load data + clean ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 One-to-one ortholog conversion (unchanged) ----
test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 Merge small donors + downsample each batch (unchanged) ----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("Downsample each batch to ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 split + merge (unchanged) ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA (dims 1:50) (unchanged) ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 Clustering + UMAP (unchanged) ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 Visualization ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# use existing labels directly: macaque = mac_5-HT_x (prefix to distinguish from mouse), mouse = 5-HT_x
mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("cells missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap heatmap (rows = mac_5-HT_x, columns = mouse 5-HT_x) ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap matrix:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(out_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_mac5HT.csv"))
cat("Overlap heatmap saved\n")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/"
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)
## =========================================================================
## Part 1: load annotated data, keep only 5-HT cells (no reclustering; use cell_type_new directly)
## =========================================================================

own <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/step5_data_save_907/all_cells_recluster/all_cells_reclustered.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

# keep only 5-HT cells, reuse existing annotations
mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT subtype composition ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## Part 2: cross-species integration (macaque x mouse, CCA dims 1:50, batches downsampled)
## =========================================================================

PAR <- list(
  variable_genes = 5000,
  npcs           = 100,
  cca_dims       = 1:50,
  nb_dims        = 1:50,
  resolution     = 0.1,
  min_cells      = 100
)

## ---- 2.1 Load data + clean ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 One-to-one ortholog conversion (unchanged) ----
test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 Merge small donors + downsample each batch (unchanged) ----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("Downsample each batch to ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 split + merge (unchanged) ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA (dims 1:50) (unchanged) ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 Clustering + UMAP (unchanged) ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 Visualization ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# use existing labels directly: macaque = mac_5-HT_x (prefix to distinguish from mouse), mouse = 5-HT_x
mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("cells missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap heatmap (rows = mac_5-HT_x, columns = mouse 5-HT_x) ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap matrix:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(out_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_mac5HT.csv"))
cat("Overlap heatmap saved\n")

# ## ---- 2.9 Save ----
# saveRDS(classFiltSeurat, paste0(out_dir, "crossSpecies_CCA_dims50_mac5HT.rds"))

In [ ]:
## ---- 2.9 Save ----
saveRDS(classFiltSeurat, paste0(save_dir, "crossSpecies_CCA_dims50_mac5HT.rds"))
writeLines(capture.output(sessionInfo()),
           paste0(save_dir, "sessionInfo_CCA_dims50_mac5HT.txt"))
message("All done")

In [ ]:
# Sankey diagram

In [ ]:
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
## =========================================================================
## 2.9 Sankey: macaque original subtypes -> (kNN mapping) -> mouse subtypes
## =========================================================================
suppressMessages({ library(ggalluvial); library(RANN); library(reshape2) })

# ---- 1. kNN label transfer in integrated.cca space ----
mac_cells <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mus_cells <- colnames(classFiltSeurat)[classFiltSeurat$species == "mouse"]

emb <- Embeddings(classFiltSeurat, "integrated.cca")[, 1:50]
emb_mac <- emb[mac_cells, , drop = FALSE]
emb_mus <- emb[mus_cells, , drop = FALSE]

mus_label <- as.character(classFiltSeurat$subtype_recl[mus_cells])

k <- 10
nn <- nn2(emb_mus, emb_mac, k = k)          # 10 nearest mouse cells for each macaque cell
pred_mouse <- vapply(seq_len(nrow(nn$nn.idx)), function(i) {
  labs <- mus_label[nn$nn.idx[i, ]]
  names(sort(table(labs), decreasing = TRUE))[1]
}, character(1))

# each macaque cell: original subtype + predicted mouse subtype
mac_lab <- as.character(classFiltSeurat$subtype_recl[mac_cells])

# ---- 2. Build macaque x mouse count matrix (mimicking your melt(NV) approach) ----
NV <- table(mac_lab, pred_mouse)

# sort by trailing number: mac_5-HT_0..10, 5-HT_0..n (avoid lexicographic order putting 10 before 2)
sort_by_num <- function(x) {
  x[order(as.numeric(sub(".*_(\\d+)$", "\\1", x)))]
}
NV <- NV[sort_by_num(rownames(NV)), sort_by_num(colnames(NV))]

mac_types <- rownames(NV)
mus_types <- colnames(NV)

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "n")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

# mapping consistency (print as well; can be a supplementary table)
map_rate <- df %>%
  group_by(macaque) %>%
  summarise(top_pred = mouse[which.max(n)],
            frac = max(n) / sum(n), .groups = "drop")
print(map_rate)
write.csv(map_rate, paste0(save_dir, "mac_to_mouse_knn_mapping.csv"),
          row.names = FALSE)

# ---- 3. Full Sankey (MetaNeighbor style) ----
p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = n)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2,
                     labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque → mouse 5-HT subtype mapping (kNN in CCA space)")

print(p_sankey)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.pdf"),
       p_sankey, width = 14, height = 7)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.png"),
       p_sankey, width = 14, height = 7, dpi = 300)

# ---- 4. Strong correspondences only (>30% of mapped cells), cleaner plot ----
df_strong <- df %>%
  group_by(macaque) %>%
  mutate(frac = n / sum(n)) %>%
  ungroup() %>%
  subset(frac > 0.3)

if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = n)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2,
                       labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque → mouse (kNN mapping, >30% of subtype)")
  print(p_sankey_strong)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.png"),
         p_sankey_strong, width = 12, height = 7, dpi = 300)
}

cat("Sankey diagram saved\n")

In [ ]:
# ---- 3. Full Sankey ----
p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = n)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  scale_x_continuous(breaks = 1:2,
                     labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque → mouse 5-HT subtype mapping (kNN in CCA space)")

print(p_sankey)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.pdf"),
       p_sankey, width = 14, height = 7)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.png"),
       p_sankey, width = 14, height = 7, dpi = 300)

# ---- 4. Sankey of strong correspondences ----
df_strong <- df %>%
  group_by(macaque) %>%
  mutate(frac = n / sum(n)) %>%
  ungroup() %>%
  subset(frac > 0.3)

if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = n)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    scale_x_continuous(breaks = 1:2,
                       labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque → mouse (kNN mapping, >30% of subtype)")
  print(p_sankey_strong)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.png"),
         p_sankey_strong, width = 12, height = 7, dpi = 300)
}

In [ ]:
# Gene plots

In [ ]:
install.packages("ggvenn")

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(ggvenn)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1028/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. Data prep (unchanged) ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

keep_pat <- "^5-HT_[0-9]+$"
is_bad <- !grepl(keep_pat, classFiltSeurat$sub_show)
if (any(is_bad)) {
  cat("cells with non-numbered subtypes removed:", sum(is_bad), fill = TRUE)
  classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!is_bad])
}

cap_first <- function(x) paste0(toupper(substr(x, 1, 1)), substr(x, 2, nchar(x)))

# ---------- 1. Find subtype markers within each species separately (unchanged) ----------
find_mk <- function(obj) {
  Idents(obj) <- "sub_show"
  mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                       min.pct = 0.1, logfc.threshold = 1,
                       test.use = "wilcox", verbose = FALSE)
  if (is.null(mk) || nrow(mk) == 0 || !"p_val_adj" %in% colnames(mk)) {
    mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                         min.pct = 0.05, logfc.threshold = 0.5,
                         test.use = "wilcox", verbose = FALSE)
  }
  stopifnot(!is.null(mk), nrow(mk) > 0, "p_val_adj" %in% colnames(mk))
  mk %>% filter(p_val_adj < 0.01) %>%      # p-value threshold 0.01
    mutate(cluster = as.character(cluster))
}

species_levels <- sort(unique(as.character(classFiltSeurat$species)))
species_levels <- c(intersect(c("macaque", "marmoset", "mouse"), species_levels),
                    setdiff(species_levels, c("macaque", "marmoset", "mouse")))
n_sp <- length(species_levels)

obj_list <- lapply(species_levels, function(sp)
  subset(classFiltSeurat, subset = species == sp))
names(obj_list) <- species_levels
mk_list <- lapply(obj_list, find_mk)



# ---------- 2. Venn diagram: marker gene sets per species ----------
gene_sets <- lapply(mk_list, function(mk) unique(mk$gene))
names(gene_sets) <- cap_first(species_levels)

all_genes <- unique(unlist(gene_sets))          # define the universe first
venn_df <- as.data.frame(                       # then build the 0/1 matrix
  sapply(gene_sets, function(g) all_genes %in% g)
)
rownames(venn_df) <- all_genes

## compute gene list for each region
region_genes <- list()
for (i in seq_len(nrow(venn_df))) {
  reg <- paste(names(venn_df)[as.logical(venn_df[i, ])], collapse = "&")
  region_genes[[reg]] <- c(region_genes[[reg]], rownames(venn_df)[i])
}

## Venn diagram itself
p_venn <- ggplot(venn_df) +
  geom_venn(aes(A = .data[[names(venn_df)[1]]],
                B = .data[[names(venn_df)[2]]],
                C = if (ncol(venn_df) >= 3) .data[[names(venn_df)[3]]] else NULL),
            fill_color = c("#EB6368", "#E8A33D", "#82ADD0")[seq_len(ncol(venn_df))],
            fill_alpha = 0.45, stroke_color = "white", stroke_size = 0.5,
            text_size = 5, set_name_size = 5) +
  coord_fixed() +
  theme_void()

ggsave(paste0(save_dir, "Fig_markerVennyouzi.pdf"), p_venn, width = 8, height = 8)
ggsave(paste0(save_dir, "Fig_markerVennyouzi.png"), p_venn, width = 8, height = 8, dpi = 300)

# ---------- 3. Export gene lists per region ----------
region_summary <- data.frame(
  region = names(region_genes),
  n_genes = lengths(region_genes)
)
write.csv(region_summary, paste0(save_dir, "Table_markerVenn_regionsyouzi.csv"),
          row.names = FALSE)
for (reg in names(region_genes)) {
  safe_reg <- gsub("[^A-Za-z0-9]+", "_", reg)
  write.csv(data.frame(gene = region_genes[[reg]]),
            paste0(save_dir, "Genes_", safe_reg, ".csv"), row.names = FALSE)
}

cat("Venn diagram saved to", save_dir, fill = TRUE)
print(region_summary)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(ggvenn)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1028/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. Data prep ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## keep only numbered subtypes (5-HT_0, 5-HT_1 ...), drop 5-HT_filtered etc.
keep_pat <- "^5-HT_[0-9]+$"
is_bad <- !grepl(keep_pat, classFiltSeurat$sub_show)
if (any(is_bad)) {
  cat("cells with non-numbered subtypes removed:", sum(is_bad), fill = TRUE)
  classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!is_bad])
}

cap_first <- function(x) paste0(toupper(substr(x, 1, 1)), substr(x, 2, nchar(x)))

# ---------- 1. Find subtype markers within each species separately (unchanged) ----------
find_mk <- function(obj) {
  Idents(obj) <- "sub_show"
  mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                       min.pct = 0.1, logfc.threshold = 1,
                       test.use = "wilcox", verbose = FALSE)
  if (is.null(mk) || nrow(mk) == 0 || !"p_val_adj" %in% colnames(mk)) {
    mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                         min.pct = 0.05, logfc.threshold = 0.5,
                         test.use = "wilcox", verbose = FALSE)
  }
  stopifnot(!is.null(mk), nrow(mk) > 0, "p_val_adj" %in% colnames(mk))
  mk %>% filter(p_val_adj < 0.01) %>%      # p-value threshold 0.01
    mutate(cluster = as.character(cluster))
}

species_levels <- sort(unique(as.character(classFiltSeurat$species)))
species_levels <- c(intersect(c("macaque", "marmoset", "mouse"), species_levels),
                    setdiff(species_levels, c("macaque", "marmoset", "mouse")))
n_sp <- length(species_levels)

obj_list <- lapply(species_levels, function(sp)
  subset(classFiltSeurat, subset = species == sp))
names(obj_list) <- species_levels
mk_list <- lapply(obj_list, find_mk)

# ---------- 2. Venn diagram: marker gene sets per species ----------
gene_sets <- lapply(mk_list, function(mk) unique(mk$gene))
names(gene_sets) <- cap_first(species_levels)

all_genes <- unique(unlist(gene_sets))          # define the universe first
venn_df <- as.data.frame(                       # then build the 0/1 matrix
  sapply(gene_sets, function(g) all_genes %in% g)
)
rownames(venn_df) <- all_genes

## compute gene list for each region
region_genes <- list()
for (i in seq_len(nrow(venn_df))) {
  reg <- paste(names(venn_df)[as.logical(venn_df[i, ])], collapse = "&")
  region_genes[[reg]] <- c(region_genes[[reg]], rownames(venn_df)[i])
}

## Venn diagram (no region counts shown, keep set names)
p_venn <- ggplot(venn_df) +
  geom_venn(aes(A = .data[[names(venn_df)[1]]],
                B = .data[[names(venn_df)[2]]],
                C = if (ncol(venn_df) >= 3) .data[[names(venn_df)[3]]] else NULL),
            fill_color = c("#EB6368", "#E8A33D", "#82ADD0")[seq_len(ncol(venn_df))],
            fill_alpha = 0.45,
            stroke_color = "white", stroke_size = 0.5,
            text_size = 0,           # do not show region counts
            set_name_size = 5) +     # keep set names (Macaque / Mouse)
  coord_fixed() +
  theme_void()

ggsave(paste0(save_dir, "Fig_markerVenn.pdf"), p_venn, width = 8, height = 8)
ggsave(paste0(save_dir, "Fig_markerVenn.png"), p_venn, width = 8, height = 8, dpi = 300)

# ---------- 3. Export gene lists per region ----------
region_summary <- data.frame(
  region = names(region_genes),
  n_genes = lengths(region_genes)
)
write.csv(region_summary, paste0(save_dir, "Table_markerVenn_regions.csv"),
          row.names = FALSE)
for (reg in names(region_genes)) {
  safe_reg <- gsub("[^A-Za-z0-9]+", "_", reg)
  write.csv(data.frame(gene = region_genes[[reg]]),
            paste0(save_dir, "Genes_", safe_reg, ".csv"), row.names = FALSE)
}

cat("Venn diagram saved to", save_dir, fill = TRUE)
print(region_summary)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(patchwork)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. Data prep ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

# ---------- 1. Cross-species DEG (macaque vs mouse, all 5-HT cells) ----------
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht   <- subset(classFiltSeurat, cells = cells_5ht)
Idents(obj_5ht) <- "species"

deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)

# ---------- 2. top30 up / down ----------
top_up <- deg %>% filter(avg_log2FC > 0) %>%
  arrange(desc(avg_log2FC)) %>% head(200) %>% rownames()
top_dn <- deg %>% filter(avg_log2FC < 0) %>%
  arrange(avg_log2FC) %>% head(200) %>% rownames()

# ---------- 3. Get real expression (3000 cells per species for faster plotting) ----------
set.seed(1)
expr <- FetchData(obj_5ht, vars = c(top_up, top_dn, "species"))
expr_s <- expr %>%
  group_by(species) %>%
  slice_sample(n = 3000) %>%
  ungroup()

to_long <- function(genes) {
  expr_s %>%
    select(species, all_of(genes)) %>%
    pivot_longer(-species, names_to = "gene", values_to = "expr") %>%
    mutate(gene   = factor(gene, levels = genes),
           species = factor(species, levels = c("mouse", "macaque")))  # mouse on the left, macaque on the right
}
long_up <- to_long(top_up)
long_dn <- to_long(top_dn)

# ---------- 4. Plot ----------
sp_fill <- c("mouse" = "#82ADD0", "macaque" = "#EB6368")

vln_panel <- function(df, title) {
  ggplot(df, aes(species, expr, fill = species)) +
    geom_violin(scale = "width", linewidth = 0.1, color = NA, na.rm = TRUE) +
    geom_boxplot(width = 0.12, outlier.shape = NA, alpha = 0.6,
                 linewidth = 0.2, color = "grey20") +
    scale_fill_manual(values = sp_fill) +
    scale_x_discrete(drop = FALSE) +
    facet_wrap(~ gene, scales = "free_y", ncol = 6) +
    labs(title = title, x = NULL, y = "Normalized expression") +
    theme_bw(base_size = 9) +
    theme(legend.position = "none",
          strip.text = element_text(face = "italic", size = 8),
          axis.text.x  = element_text(size = 8),
          plot.title   = element_text(face = "bold", size = 12))
}

p_up <- vln_panel(long_up, "Top 200 up in macaque (vs mouse)")
p_dn <- vln_panel(long_dn, "Top 200 down in macaque (up in mouse)")

pdf(paste0(save_dir, "Fig_5HT_top50_up_down_violin.pdf"), width = 12, height = 18)
print(p_up / p_dn + plot_layout(heights = c(1, 1)))
dev.off()

png(paste0(save_dir, "Fig_5HT_top200_up_down_violin.png"),
    width = 12, height = 18, units = "in", res = 300)
print(p_up / p_dn + plot_layout(heights = c(1, 1)))
dev.off()

cat("Violin plot saved to", save_dir, fill = TRUE)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(patchwork)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

GENE <- "NXPH2"     # change the gene here

# ---------- 0. Data prep ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht   <- subset(classFiltSeurat, cells = cells_5ht)
stopifnot(GENE %in% rownames(obj_5ht))

sort_by_num <- function(x) {
  n <- suppressWarnings(as.numeric(sub(".*_(\\d+)$", "\\1", x)))
  x[order(n, na.last = TRUE)]
}

# ---------- 1. Get expression (data = log-normalized) and raw counts ----------
df <- FetchData(obj_5ht, vars = c(GENE, "species", "sub_show"))
colnames(df)[1] <- "expr"
cnt <- FetchData(obj_5ht, vars = GENE, layer = "counts")
df$count <- cnt[rownames(df), 1]
df$species  <- factor(as.character(df$species), levels = c("macaque", "mouse"))
df$sub_show <- factor(as.character(df$sub_show),
                      levels = sort_by_num(unique(as.character(df$sub_show))))
df$express  <- df$count > 0

# ---------- 2. Violin plot: subtypes within each species ----------
sp_fill <- c("macaque" = "#EB6368", "mouse" = "#82ADD0")

p_vln <- ggplot(df, aes(sub_show, expr, fill = species)) +
  geom_violin(scale = "width", linewidth = 0.1, color = NA) +
  scale_fill_manual(values = sp_fill) +
  facet_wrap(~ species, nrow = 1, scales = "free") +
  labs(title = paste0(GENE, " expression across 5-HT subclasses"),
       subtitle = "Raw UMI counts", x = NULL, y = "Raw count") +
  theme_classic(base_size = 11) +
  theme(legend.position = "none",
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1))

# ---------- 3. Expression proportion: fraction of expressing cells within each subtype (between-species comparison) ----------
pct_df <- df %>%
  group_by(species, sub_show) %>%
  summarise(pct = mean(express), n = n(), .groups = "drop")

## Fisher's exact test per shared subtype
subs_mac  <- sort_by_num(unique(as.character(df$sub_show[df$species == "macaque"])))
subs_mus  <- sort_by_num(unique(as.character(df$sub_show[df$species == "mouse"])))
subs_both <- intersect(subs_mac, subs_mus)

sub_stat <- lapply(subs_both, function(s) {
  d <- df[as.character(df$sub_show) == s, ]
  if (n_distinct(d$species) < 2) return(NULL)
  tb <- table(d$species, d$express)
  if (nrow(tb) < 2 || ncol(tb) < 2) return(NULL)
  ft <- fisher.test(tb)
  data.frame(sub_show = s,
             pct_mac = mean(d$express[d$species == "macaque"]),
             pct_mus = mean(d$express[d$species == "mouse"]),
             p_val = ft$p.value)
})
sub_stat <- bind_rows(sub_stat)
if (nrow(sub_stat) > 0) {
  sub_stat$p_adj <- p.adjust(sub_stat$p_val, method = "BH")
  sub_stat$star <- cut(sub_stat$p_adj,
                       breaks = c(-1, 0.001, 0.01, 0.05, 1),
                       labels = c("***", "**", "*", "ns"))
}

lab_df <- pct_df %>%
  left_join(sub_stat %>% select(sub_show, star), by = "sub_show") %>%
  mutate(lab = paste0(round(pct * 100), "%"))

p_pct <- ggplot(pct_df, aes(sub_show, pct, fill = species)) +
  geom_col(position = position_dodge(0.85), width = 0.75, color = NA) +
  geom_text(data = lab_df,
            aes(label = lab), position = position_dodge(0.85),
            vjust = -0.4, size = 3) +
  geom_text(data = sub_stat,
            aes(x = sub_show, y = max(pct_mac, pct_mus) + 0.12,
                label = star), inherit.aes = FALSE, size = 4) +
  scale_fill_manual(values = sp_fill) +
  scale_y_continuous(labels = scales::percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = paste0(GENE, ": fraction of expressing cells"),
       subtitle = "Fisher exact test, BH-adjusted", x = NULL, y = "% of cells") +
  theme_classic(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# ---------- 4. Overall proportion: between-species comparison across all 5-HT cells ----------
overall <- df %>%
  group_by(species) %>%
  summarise(pct = mean(express), n = n(), .groups = "drop")
tab_all <- table(df$species, df$express)
ft_all  <- fisher.test(tab_all)
p_all   <- chisq.test(tab_all)$p.value

p_ov <- ggplot(overall, aes(species, pct, fill = species)) +
  geom_col(width = 0.6, color = NA) +
  geom_text(aes(label = paste0(round(pct * 100, 1), "%\n(n=", n, ")")),
            vjust = -0.3, size = 3.5) +
  scale_fill_manual(values = sp_fill) +
  scale_y_continuous(labels = scales::percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = paste0(GENE, ": overall expression in 5-HT cells"),
       subtitle = paste0("Fisher p = ", signif(ft_all$p.value, 3),
                         " | chi-sq p = ", signif(p_all, 3)),
       x = NULL, y = "% of 5-HT cells") +
  theme_classic(base_size = 11) +
  theme(legend.position = "none")

# ---------- 5. Plot + stats table ----------
p_all_fig <- (p_vln / p_pct) | p_ov
p_all_fig <- p_all_fig + plot_layout(widths = c(3, 1))

ggsave(paste0(save_dir, "Fig_", GENE, "_subclass_violin_pct.pdf"),
       p_all_fig, width = 14, height = 10)
ggsave(paste0(save_dir, "Fig_", GENE, "_subclass_violin_pct.png"),
       p_all_fig, width = 14, height = 10, dpi = 300)

write.csv(sub_stat, paste0(save_dir, "Table_", GENE, "_pct_by_subclass.csv"),
          row.names = FALSE)
write.csv(overall, paste0(save_dir, "Table_", GENE, "_pct_overall.csv"),
          row.names = FALSE)

cat("==== ", GENE, " overall expression proportion ====", fill = TRUE)
print(overall)
cat("Fisher p =", ft_all$p.value, " | chi-sq p =", p_all, fill = TRUE)
cat("Plot and table saved to", save_dir, fill = TRUE)

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 0. Data prep ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## keep only 5-HT cells (all subtypes merged)
## use cells = for subsetting, to avoid FetchData parsing issues with subset expressions
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

## check cell counts and species distribution
print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. All 5-HT cells: macaque vs mouse ----
## avg_log2FC > 0 = up in macaque; < 0 = up in mouse
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. Volcano plot ----
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),  # avoid Inf from p = 0
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(5),
  df %>% filter(sig == "down") %>% pick_spread(5)
)

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    size = 3.5, color = "black", fontface = "italic",
    box.padding = 0.8, point.padding = 0.5,
    force = 5, force_pull = 0.2, direction = "both",
    max.overlaps = Inf, min.segment.length = 0, seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13) +
  theme(plot.margin = margin(20, 40, 20, 20))

# ---- 3. Plot + export ----
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 7, height = 6)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 7, height = 6, dpi = 300)

## use dplyr::select to avoid AnnotationDbi masking
deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("Volcano plot saved to", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 0. Data prep ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## keep only 5-HT cells (all subtypes merged)
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

## check cell counts and species distribution
print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. All 5-HT cells: macaque vs mouse ----
## avg_log2FC > 0 = up in macaque; < 0 = up in mouse
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. Volcano plot ----
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),  # avoid Inf from p = 0
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## labels: top5 per side + force-label NXPH2
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(5),
  df %>% filter(sig == "down") %>% pick_spread(5)
)

nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("NOTE: NXPH2 is not in this DEG result (may have been filtered out by min.pct = 0.1)")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  ## highlight NXPH2 with a large purple point
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 2.5, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    size = 3.5, color = "black", fontface = "italic",
    box.padding = 0.8, point.padding = 0.5,
    force = 5, force_pull = 0.2, direction = "both",
    max.overlaps = Inf, min.segment.length = 0, seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13) +
  theme(plot.margin = margin(20, 40, 20, 20))

# ---- 3. Plot + export ----
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 7, height = 6)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 7, height = 6, dpi = 300)

deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("Volcano plot saved to", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## check whether Arial is available (often missing on Linux)
## if 0 rows are returned, it is not installed; use "Liberation Sans" instead
system("fc-list | grep -i arial")

# ---- 0. Data prep ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## keep only 5-HT cells (all subtypes merged)
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. All 5-HT cells: macaque vs mouse ----
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. Volcano plot ----
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## labels: top5 per side + force-label NXPH2
## labels: top6 per side (2 smallest p + 4 largest |FC|) + force-label NXPH2
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(6),
  df %>% filter(sig == "down") %>% pick_spread(6)
)
nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("NOTE: NXPH2 is not in this DEG result (may have been filtered out by min.pct = 0.1)")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

## font setting: Arial 8pt
ft <- "Arial"   # if fc-list does not find Arial, change to "Liberation Sans"

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  ## highlight NXPH2 with a large purple point
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 2.5, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    ## labels: Arial 8pt (ggplot text size is in mm; 8pt ~ 2.82mm)
    family = ft, size = 8 / ggplot2::.pt, color = "black", fontface = "italic",
    box.padding = 0.8, point.padding = 0.5,
    force = 5, force_pull = 0.2, direction = "both",
    max.overlaps = Inf, min.segment.length = 0, seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  ## theme: Arial 8pt
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text            = element_text(family = ft, size = 8),
    plot.title      = element_text(family = ft, size = 8),
    axis.title      = element_text(family = ft, size = 8),
    axis.text       = element_text(family = ft, size = 8),
    legend.text     = element_text(family = ft, size = 8),
    legend.title    = element_text(family = ft, size = 8),
    plot.margin     = margin(20, 40, 20, 20)
  )

# ---- 3. Plot + export ----
## use cairo_pdf device for PDF; more reliable font embedding
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 7, height = 6, device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 7, height = 6, dpi = 300)

deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("Volcano plot saved to", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## check whether Arial is available (often missing on Linux)
## if 0 rows are returned, it is not installed; change ft below to "Liberation Sans"
system("fc-list | grep -i arial")

# ---- 0. Data prep ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## keep only 5-HT cells (all subtypes merged)
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. All 5-HT cells: macaque vs mouse ----
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. Volcano plot ----
pick_spread <- function(sub, n = 6) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),  # avoid Inf from p = 0
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## labels: top6 per side (2 smallest p + 4 largest |FC|) + force-label NXPH2
## labels: only top1 per side (smallest p.adj) + force-label NXPH2
## labels: only top1 per side (smallest p.adj); no other labels
## labels: only top1 per side; on p-value ties take the largest |FC|, no ties kept
lab <- bind_rows(
  df %>% filter(sig == "up") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1),
  df %>% filter(sig == "down") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1)
)

nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("NOTE: NXPH2 is not in this DEG result (may have been filtered out by min.pct = 0.1)")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

## font setting: Arial 8pt
ft <- "Arial"   # if fc-list does not find Arial, change to "Liberation Sans"

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.35, alpha = 0.6) +
  ## highlight NXPH2 with a large purple point
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 1.6, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
    geom_text_repel(
    data = lab, aes(label = gene),
    family = ft, size = 8 / ggplot2::.pt, color = "black", fontface = "italic",
    ## shorten further: stronger pull, weaker repulsion, smaller spacing
    box.padding = 0.4,
    point.padding = 0.3,
    force = 1.5, force_pull = 3,     # original force = 3, force_pull = 2
    direction = "both",
    max.overlaps = Inf,
    min.segment.length = 0,
    seed = 42,
    max.time = 3, max.iter = 1000000,
    segment.size = 0.3, segment.alpha = 0.7,   # thinner lines (original 0.35)
    segment.curvature = -0.1, segment.ncp = 3, segment.angle = 20
  ) +
  ## increase expansion space
  coord_cartesian(clip = "off", expand = TRUE) +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  ## theme: Arial 8pt
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text            = element_text(family = ft, size = 8),
    plot.title      = element_text(family = ft, size = 8),
    axis.title      = element_text(family = ft, size = 8),
    axis.text       = element_text(family = ft, size = 8),
    legend.text     = element_text(family = ft, size = 8),
    legend.title    = element_text(family = ft, size = 8),
    ## widen margins to leave room for staggered labels
        ## small figure needs small margins (a 6.5cm figure cannot fit the original margin(40, 80, 40, 40))
    plot.margin     = margin(6, 10, 6, 6)
  )

# ---- 3. Plot + export ----
## height 6 -> 8, more vertical space for the dense top area
# ---- 3. Plot + export ----
## width 6.5, height 6
# ---- 3. Plot + export ----
## height 6cm x width 6.5cm (standard single-column journal figure size)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 6.5, height = 6, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 6.5, height = 6, units = "cm", dpi = 600)  # small figure: suggest dpi 600
deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("Volcano plot saved to", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## check whether Arial is available (often missing on Linux)
## if 0 rows are returned, it is not installed; change ft below to "Liberation Sans"
system("fc-list | grep -i arial")

# ---- 0. Data prep ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## keep only 5-HT cells (all subtypes merged)
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. All 5-HT cells: macaque vs mouse ----
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. Volcano plot ----
pick_spread <- function(sub, n = 6) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),
         sig = case_when(y_raw > 50 & avg_log2FC >  3 ~ "up",    # ★ 1 → 3
                         y_raw > 50 & avg_log2FC < -3 ~ "down",  # ★ -1 → -3
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## labels: top6 per side (2 smallest p + 4 largest |FC|) + force-label NXPH2
## labels: only top1 per side (smallest p.adj) + force-label NXPH2
## labels: only top1 per side (smallest p.adj); no other labels
## labels: only top1 per side; on p-value ties take the largest |FC|, no ties kept
lab <- bind_rows(
  df %>% filter(sig == "up") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1),
  df %>% filter(sig == "down") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1)
)

nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("NOTE: NXPH2 is not in this DEG result (may have been filtered out by min.pct = 0.1)")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

## font setting: Arial 8pt
ft <- "Arial"   # if fc-list does not find Arial, change to "Liberation Sans"

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.35, alpha = 0.6) +
  ## highlight NXPH2 with a large purple point
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 1.6, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-3, 3), linetype = "dashed", color = "grey50") +  # ★ ±1 → ±3
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
    geom_text_repel(
    data = lab, aes(label = gene),
    family = ft, size = 8 / ggplot2::.pt, color = "black", fontface = "italic",
    ## shorten further: stronger pull, weaker repulsion, smaller spacing
    box.padding = 0.4,
    point.padding = 0.3,
    force = 1.5, force_pull = 3,     # original force = 3, force_pull = 2
    direction = "both",
    max.overlaps = Inf,
    min.segment.length = 0,
    seed = 42,
    max.time = 3, max.iter = 1000000,
    segment.size = 0.3, segment.alpha = 0.7,   # thinner lines (original 0.35)
    segment.curvature = -0.1, segment.ncp = 3, segment.angle = 20
  ) +
  ## increase expansion space
  coord_cartesian(clip = "off", expand = TRUE) +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  ## theme: Arial 8pt
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text            = element_text(family = ft, size = 8),
    plot.title      = element_text(family = ft, size = 8),
    axis.title      = element_text(family = ft, size = 8),
    axis.text       = element_text(family = ft, size = 8),
    legend.text     = element_text(family = ft, size = 8),
    legend.title    = element_text(family = ft, size = 8),
    ## widen margins to leave room for staggered labels
        ## small figure needs small margins (a 6.5cm figure cannot fit the original margin(40, 80, 40, 40))
    plot.margin     = margin(6, 10, 6, 6)
  )

# ---- 3. Plot + export ----
## height 6 -> 8, more vertical space for the dense top area
# ---- 3. Plot + export ----
## width 6.5, height 6
# ---- 3. Plot + export ----
## height 6cm x width 6.5cm (standard single-column journal figure size)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse2.pdf"),
       p, width = 6.5, height = 6, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse2.png"),
       p, width = 6.5, height = 6, units = "cm", dpi = 600)  # small figure: suggest dpi 600
deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse2.csv"),
          row.names = FALSE)

print(p)
cat("Volcano plot saved to", save_dir, "\n")

In [ ]:
# ---- 4. NXPH2 expression violin plot: macaque vs mouse ----
## extract NXPH2 expression data
df_nxph2 <- FetchData(obj_5ht, vars = c("NXPH2", "species"))
df_nxph2$species <- factor(df_nxph2$species, levels = c("macaque", "mouse"))

p_nx <- ggplot(df_nxph2, aes(x = species, y = NXPH2, fill = species)) +
  ## plain violin: no boxplot, no median line, no points
  geom_violin(scale = "width", trim = TRUE, adjust = 1,
              linewidth = 0.3, color = "black") +
  scale_fill_manual(values = c(macaque = "#EB6368", mouse = "#82ADD0")) +
  labs(x = NULL, y = "NXPH2 expression", fill = NULL) +
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text        = element_text(family = ft, size = 8),
    axis.text   = element_text(family = ft, size = 8),
    axis.title  = element_text(family = ft, size = 8),
    legend.text = element_text(family = ft, size = 8),
    legend.position = "none",          # colors already match the x-axis; no legend needed
    plot.margin = margin(6, 10, 6, 6)
  )

ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.pdf"),
       p_nx, width = 4, height = 4.5, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.png"),
       p_nx, width = 4, height = 4.5, units = "cm", dpi = 600)

print(p_nx)

In [ ]:
# ---- 4. NXPH2 expression violin plot: macaque vs mouse ----
## extract NXPH2 expression data
df_nxph2 <- FetchData(obj_5ht, vars = c("NXPH2", "species"))
df_nxph2$species <- factor(df_nxph2$species, levels = c("macaque", "mouse"))

p_nx <- ggplot(df_nxph2, aes(x = species, y = NXPH2, fill = species)) +
  ## plain violin: no boxplot, no median line, no points
  geom_violin(scale = "width", trim = TRUE, adjust = 1,
              linewidth = 0.3, color = "black") +
  scale_fill_manual(values = c(macaque = "#EB6368", mouse = "#82ADD0")) +
  labs(x = NULL, y = "NXPH2 expression", fill = NULL) +
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text        = element_text(family = ft, size = 8),
    axis.text   = element_text(family = ft, size = 8),
    axis.title  = element_text(family = ft, size = 8),
    legend.text = element_text(family = ft, size = 8),
    legend.position = "none",          # colors already match the x-axis; no legend needed
    plot.margin = margin(6, 10, 6, 6)
  )

ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.pdf"),
       p_nx, width = 4, height = 4.5, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.png"),
       p_nx, width = 4, height = 4.5, units = "cm", dpi = 600)

print(p_nx)

In [ ]:
## 1) NXPH2 info itself
deg %>% filter(gene == "NXPH2") %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)))

## 2) rank by p.adj among "significantly up" genes (y>50 and FC>1)
deg %>%
  filter(p_val_adj < 1e-50, avg_log2FC > 1) %>%
  arrange(p_val_adj) %>%
  mutate(rank_by_p = row_number(),
         rank_by_fc = rank(-avg_log2FC)) %>%
  filter(gene == "NXPH2")

## 3) rank by p.adj among "significantly down" genes (y>50 and FC<-1)
deg %>%
  filter(p_val_adj < 1e-50, avg_log2FC < -1) %>%
  arrange(p_val_adj) %>%
  mutate(rank_by_p = row_number(),
         rank_by_fc = rank(avg_log2FC)) %>%
  filter(gene == "NXPH2")

## 4) overall rank by p.adj in the entire DEG table, regardless of direction
deg %>% arrange(p_val_adj) %>%
  mutate(rank_total = row_number()) %>%
  filter(gene == "NXPH2")

In [ ]:
deg %>% filter(gene == "NXPH2") %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)))

In [ ]:
## NXPH2 rank among significantly up genes (no select dependency)
res <- deg %>%
  filter(p_val_adj < 1e-50, avg_log2FC > 1) %>%
  arrange(p_val_adj) %>%
  mutate(rank_by_p  = row_number(),           # rank by p.adj ascending
         rank_by_fc = rank(-avg_log2FC)) %>%  # rank by FC descending
  filter(gene == "NXPH2")

print(res[, c("gene", "avg_log2FC", "p_val_adj", "rank_by_p", "rank_by_fc")])

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat)
  library(ComplexHeatmap); library(circlize); library(grid); library(dplyr)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

# ---- 1. Single macaque vs mouse comparison (logfc.threshold = 0, keep all genes) ----
Idents(classFiltSeurat) <- "species"
mk <- FindMarkers(classFiltSeurat, assay = "RNA",
                  ident.1 = "macaque", ident.2 = "mouse",
                  min.pct = 0.05, logfc.threshold = 0,
                  test.use = "wilcox", verbose = FALSE)

sig <- mk$p_val_adj < 0.05
genes_mac <- rownames(mk)[sig & mk$avg_log2FC >  0.5]   # enriched in macaque
genes_mus <- rownames(mk)[sig & mk$avg_log2FC < -0.5]   # enriched in mouse
# conserved/shared: not significantly different + expressed in both species
shared <- rownames(mk)[!sig & mk$pct.1 > 0.25 & mk$pct.2 > 0.25]

cat("Shared:", length(shared), " macaque-specific:", length(genes_mac),
    " mouse-specific:", length(genes_mus), "\n")

# ---- 2. Three-block classification ----
genes_use <- c(shared, genes_mac, genes_mus)
gene2stat <- setNames(c(rep("Shared", length(shared)),
                        rep("Macaque-specific", length(genes_mac)),
                        rep("Mouse-specific", length(genes_mus))),
                      genes_use)
n_stat <- c("Shared" = length(shared),
            "Macaque-specific" = length(genes_mac),
            "Mouse-specific" = length(genes_mus))

# ---- 3. Expression matrix (genes x cells -> z-score) ----
mat <- as.matrix(GetAssayData(classFiltSeurat, assay = "RNA",
                              slot = "data")[genes_use, , drop = FALSE])
z <- t(scale(t(mat)))
z[is.na(z)] <- 0
z[z > 2] <- 2; z[z < -2] <- -2

cell_sp <- as.character(classFiltSeurat$species)

# ---- 4. Heatmap: rows = cells (mouse on top, macaque on bottom), columns = three gene blocks ----
stat_cols <- c("Shared" = "grey35",
               "Macaque-specific" = "#EB6368",
               "Mouse-specific"   = "#82ADD0")
col_stat <- factor(gene2stat[genes_use], levels = names(stat_cols))
col_titles <- sprintf("%s\nn=%d", names(n_stat), n_stat)

ht <- Heatmap(t(z),                              # transpose: rows = cells, columns = genes
  name = "z-score",
  col = colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B")),
  row_split = factor(cell_sp, levels = c("mouse", "macaque")),
  row_title = c("mouse", "macaque"),
  row_title_gp = gpar(fontsize = 12, fontface = "bold"),
  cluster_row_slices = TRUE, cluster_rows = TRUE,
  show_row_dend = FALSE, show_row_names = FALSE,
  column_split = col_stat,
  cluster_column_slices = FALSE, cluster_columns = FALSE,
  show_column_names = FALSE,
  column_title = col_titles,
  column_title_gp = gpar(fontsize = 11, fontface = "bold"),
  border = FALSE, rect_gp = gpar(col = NA),
  right_annotation = rowAnnotation(
    species = cell_sp,
    col = list(species = c(macaque = "#EB6368", mouse = "#82ADD0")),
    show_annotation_name = FALSE,
    annotation_legend_param = list(species = list(title = "species"))),
  top_annotation = columnAnnotation(
    Conservation = col_stat,
    col = list(Conservation = stat_cols),
    show_annotation_name = FALSE,
    annotation_legend_param = list(Conservation = list(title = "Conservation"))),
  use_raster = TRUE, raster_quality = 3,
  width = unit(16, "cm"), height = unit(14, "cm")
)

pdf(paste0(save_dir, "Fig_5HT_shared_specific_genes_cells.pdf"),
    width = 12, height = 10)
draw(ht)
dev.off()

png(paste0(save_dir, "Fig_5HT_shared_specific_genes_cells.png"),
    width = 12, height = 10, units = "in", res = 300)
draw(ht)
dev.off()

# ---- 5. Output gene counts and lists ----
gene_table <- data.frame(gene = genes_use,
                         conservation = gene2stat[genes_use],
                         row.names = NULL)
write.csv(gene_table, paste0(save_dir, "Table_5HT_shared_specific_genes.csv"),
          row.names = FALSE)
write.csv(data.frame(category = names(n_stat), n = as.integer(n_stat)),
          paste0(save_dir, "Table_5HT_shared_specific_geneCounts.csv"),
          row.names = FALSE)
cat("Plot and gene table saved to", save_dir, "\n")

In [ ]:
# GO analysis

In [ ]:
library(dplyr)
library(ggplot2)
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"
## ============================================================
## Diagnostic 1: number of genes passing filters per subtype across threshold combinations
## ============================================================
threshold_grid <- expand.grid(
  FC  = c(0.25, 0.5, 0.75, 1.0),
  PCT = c(0.1, 0.2, 0.3, 0.5),
  stringsAsFactors = FALSE
)

diag_pass <- lapply(split(threshold_grid, seq_len(nrow(threshold_grid))), function(th) {
  mk_mac %>%
    filter(p_val_adj < 0.05, avg_log2FC > th$FC, pct.1 > th$PCT) %>%
    count(cluster, name = "n_pass") %>%
    mutate(FC = th$FC, PCT = th$PCT)
}) %>% bind_rows()

# wide table: rows = subtypes, columns = threshold combinations
diag_wide <- diag_pass %>%
  mutate(combo = paste0("FC>", FC, "_pct>", PCT)) %>%
  select(cluster, combo, n_pass) %>%
  pivot_wider(names_from = combo, values_from = n_pass, values_fill = 0) %>%
  arrange(cluster)

print(diag_wide, n = Inf)
write.csv(diag_wide, paste0(save_dir, "Table_threshold_diagnostic.csv"),
          row.names = FALSE)

## ============================================================
## Diagnostic 2: distribution of avg_log2FC per subtype (decide the FC cutoff)
## ============================================================
fc_dist <- mk_mac %>%
  filter(p_val_adj < 0.05) %>%
  group_by(cluster) %>%
  summarise(
    n_sig      = n(),
    fc_median  = round(median(avg_log2FC), 2),
    fc_top10   = round(sort(avg_log2FC, decreasing = TRUE)[min(10, n())], 2),
    fc_top30   = round(sort(avg_log2FC, decreasing = TRUE)[min(30, n())], 2),
    fc_top50   = round(sort(avg_log2FC, decreasing = TRUE)[min(50, n())], 2),
    .groups = "drop"
  ) %>%
  arrange(cluster)

print(fc_dist, n = Inf)
# fc_top10/30/50 = fold change of the gene ranked 10/30/50 by FC within that subtype
# if fc_top50 is only 0.4, many genes in that subtype's "top 50" have FC < 0.5

## ============================================================
## Diagnostic 3: what TOP_N? look at the cumulative distribution after FC>0.5 & pct>0.3
## ============================================================
n_after <- mk_mac %>%
  filter(p_val_adj < 0.05, avg_log2FC > 0.5, pct.1 > 0.3) %>%
  count(cluster, name = "n_after_filter") %>%
  arrange(cluster)

print(n_after, n = Inf)
hist(n_after$n_after_filter, breaks = 20,
     main = "distribution of genes passing FC>0.5 & pct>0.3 per subtype",
     xlab = "number of genes")

## ============================================================
## Diagnostic 4 (quick decision): find a threshold "not too strict, not too loose"
## ============================================================
# goal: most subtypes retain 30-100 genes after filtering (sweet spot for GO enrichment)
target_check <- diag_pass %>%
  group_by(FC, PCT) %>%
  summarise(
    n_subtype     = n(),
    min_genes     = min(n_pass),
    median_genes  = round(median(n_pass)),
    n_below_30    = sum(n_pass < 30),    # subtypes with too few genes left (unstable GO)
    n_above_100   = sum(n_pass > 100),   # subtypes with too many genes left (TOP_N truncation wasteful)
    .groups = "drop"
  ) %>%
  arrange(desc(n_below_30 == 0), median_genes)   # prioritize combos with no subtype below 30

print(target_check, n = Inf)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")
BiocManager::install("org.Mm.eg.db")

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db); library(org.Mm.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"

## ============================================================
## 0. Define two analysis units (change here to switch gene sets)
## ============================================================
analyses <- list(
  list(label    = "macaque_specific",
       genes    = panel %>% filter(status == "Macaque-specific") %>% pull(gene),
       bg       = unique(mk_mac$gene),        # all genes tested by macaque FindAllMarkers
       OrgDb    = org.Hs.eg.db),
  list(label    = "mouse_specific",
       genes    = panel %>% filter(status == "Mouse-specific") %>% pull(gene),
       bg       = unique(mk_mus$gene),        # genes tested in mouse
       OrgDb    = org.Mm.eg.db)
)

## ============================================================
## 1. Enrichment + plotting function
## ============================================================
run_one <- function(gene_set, bg_genes, OrgDb, label) {
  cat("\n========== ", label, " ==========\n")
  cat("gene set size:", length(gene_set), "  background set:", length(bg_genes), "\n")

  # key point: convert gene set and background with the same bitr so mapping rates are comparable
  gene_map <- bitr(gene_set, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = OrgDb)
  bg_map   <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = OrgDb)

  cat("gene set mapping rate:", nrow(gene_map), "/", length(gene_set),
      " (", round(100 * nrow(gene_map) / length(gene_set), 1), "%)\n", sep = "")
  cat("background mapping rate:", nrow(bg_map), "/", length(bg_genes),
      " (", round(100 * nrow(bg_map) / length(bg_genes), 1), "%)\n", sep = "")

  ego_bp <- enrichGO(gene          = gene_map$ENTREZID,
                     universe      = bg_map$ENTREZID,
                     OrgDb         = OrgDb,
                     keyType       = "ENTREZID",
                     ont           = "BP",
                     pAdjustMethod = "BH",
                     pvalueCutoff  = 0.05,
                     qvalueCutoff  = 0.2,
                     readable      = TRUE)

  df <- as.data.frame(ego_bp)
  cat("enriched to ", nrow(df), " BP pathways\n")

  write.csv(df, paste0(save_dir, "GO_BP_", label, ".csv"), row.names = FALSE)

  if (nrow(df) > 0) {
    p_dot <- dotplot(ego_bp, showCategory = 20, font.size = 9) +
      ggtitle(paste0("GO BP: ", label))
    ggsave(paste0(save_dir, "GO_BP_", label, "_dotplot.pdf"),
           p_dot, width = 8, height = 7)
    ggsave(paste0(save_dir, "GO_BP_", label, "_dotplot.png"),
           p_dot, width = 8, height = 7, dpi = 300)

    p_bar <- barplot(ego_bp, showCategory = 20, font.size = 9) +
      ggtitle(paste0("GO BP: ", label))
    ggsave(paste0(save_dir, "GO_BP_", label, "_barplot.pdf"),
           p_bar, width = 8, height = 7)

    p_net <- cnetplot(ego_bp, showCategory = 8, circular = FALSE,
                      colorEdge = TRUE, node_label = "gene",
                      cex_label_category = 0.9, cex_label_gene = 0.7)
    ggsave(paste0(save_dir, "GO_BP_", label, "_cnetplot.pdf"),
           p_net, width = 11, height = 9)

    ego_s <- pairwise_termsim(ego_bp)
    p_tree <- treeplot(ego_s, showCategory = 20)
    ggsave(paste0(save_dir, "GO_BP_", label, "_treeplot.pdf"),
           p_tree, width = 9, height = 7)

    print(p_dot)
  }
  invisible(df)
}

## ============================================================
## 2. Run the two species-specific sets
## ============================================================
go_results <- lapply(analyses, function(a)
  run_one(a$genes, a$bg, a$OrgDb, a$label))
names(go_results) <- sapply(analyses, `[[`, "label")

In [ ]:
suppressMessages({ library(ggplot2); library(dplyr) })

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"

## ============================================================
## read results + filter (consistent with the previously defined non-raphe blacklist)
## ============================================================
go_mac <- read.csv(paste0(save_dir, "GO_BP_macaque_specific.csv"))

drop_terms <- c("ossification", "osteoblast differentiation",
                "epithelial cell proliferation",
                "positive regulation of epithelial cell proliferation",
                "morphogenesis of a branching epithelium",
                "muscle cell proliferation",
                "response to lipopolysaccharide",
                "response to molecule of bacterial origin")

go_show <- go_mac %>% filter(!Description %in% drop_terms)

TOP_N <- 10

plot_df <- go_show %>%
  slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
  mutate(neglogP = -log10(p.adjust),
         label   = paste0(Description, " (", ID, ")"),
         # most significant terms on top
         label   = factor(label, levels = rev(label)))

## ============================================================
## barplot
## ============================================================
p_bar <- ggplot(plot_df, aes(x = neglogP, y = label, fill = neglogP)) +
  geom_bar(stat = "identity", width = 0.7, color = NA) +
  scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
                       midpoint = max(plot_df$neglogP) / 2,
                       name = "-log10(Padj)") +
  scale_x_continuous(expand = expansion(mult = c(0, 0.1))) +
  labs(title = "Top GO Biological Processes",
       x = "-log10(Padj)", y = NULL) +
  theme_classic(base_size = 10) +
  theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 12),
        axis.text.y = element_text(size = 9, color = "black"),
        axis.text.x = element_text(size = 9, color = "black"),
        axis.line.y = element_blank(),
        axis.ticks.y = element_blank(),
        legend.position = "right",
        legend.title = element_text(size = 9),
        legend.text  = element_text(size = 8),
        legend.key.size = unit(0.4, "cm"))

print(p_bar)

ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_custom.png"),
       p_bar, width = 8, height = 5.5, dpi = 300)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_custom.pdf"),
       p_bar, width = 8, height = 5.5)

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})
select <- dplyr::select

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1013/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## ============================================================
## 0. Prep: macaque-specific gene set + rank by FC
## ============================================================
mac_specific <- panel %>% filter(status == "Macaque-specific") %>% pull(gene)

# rank each gene by its best avg_log2FC in mk_mac (a gene may appear in multiple subtypes; take the max)
fc_rank <- mk_mac %>%
  filter(gene %in% mac_specific) %>%
  group_by(gene) %>%
  summarise(max_fc = max(avg_log2FC), .groups = "drop") %>%
  arrange(desc(max_fc))

bg_genes <- unique(mk_mac$gene)

cat("total macaque-specific genes:", nrow(fc_rank), "\n")

## ============================================================
## 1. GO function (thresholds as before)
## ============================================================
run_go <- function(gene_set, bg_genes, top_n) {
  gene_map <- bitr(gene_set, fromType = "SYMBOL", toType = "ENTREZID",
                   OrgDb = org.Hs.eg.db)
  bg_map   <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID",
                   OrgDb = org.Hs.eg.db)

  ego <- enrichGO(gene          = gene_map$ENTREZID,
                  universe      = bg_map$ENTREZID,
                  OrgDb         = org.Hs.eg.db,
                  keyType       = "ENTREZID",
                  ont           = "BP",
                  pAdjustMethod = "BH",
                  pvalueCutoff  = 0.05,
                  qvalueCutoff  = 0.2,
                  readable      = TRUE)

  df <- as.data.frame(ego) %>% mutate(top_n = top_n)
  cat(sprintf("Top%-4d | genes %4d (mapped %d) | enriched %d BP\n",
              top_n, length(gene_set), nrow(gene_map), nrow(df)))
  df
}

## ============================================================
## 2. Loop Top 10,20,...,100 separately
## ============================================================
top_n_vec <- seq(10, 100, by = 10)

go_list <- lapply(top_n_vec, function(nn) {
  genes_n <- fc_rank %>% slice_head(n = min(nn, nrow(.))) %>% pull(gene)
  run_go(genes_n, bg_genes, nn)
})
names(go_list) <- paste0("Top", top_n_vec)

# summary table: number of pathways enriched per N
summary_df <- data.frame(
  top_n = top_n_vec,
  n_genes = top_n_vec,
  n_BP = sapply(go_list, nrow)
)
print(summary_df)
write.csv(summary_df, paste0(save_dir, "GO_topN_summary.csv"), row.names = FALSE)

# store full results for each N separately
for (nm in names(go_list)) {
  write.csv(go_list[[nm]],
            paste0(save_dir, "GO_BP_macaque_specific_", nm, ".csv"),
            row.names = FALSE)
}

## ============================================================
## 3. Quick comparison: enriched pathways vs N (look for saturation)
## ============================================================
p_sat <- ggplot(summary_df, aes(top_n, n_BP)) +
  geom_line(color = "#B2182B", linewidth = 0.8) +
  geom_point(size = 2, color = "#B2182B") +
  scale_x_continuous(breaks = top_n_vec) +
  labs(x = "Top N genes (ranked by avg_log2FC)",
       y = "Number of enriched GO BP terms",
       title = "GO enrichment saturation across top-N thresholds") +
  theme_bw(base_size = 10) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 11))

print(p_sat)
ggsave(paste0(save_dir, "GO_topN_saturation.png"), p_sat,
       width = 7, height = 4.5, dpi = 300)

# pull out the significance trajectory of GPCR terms across N (core terms of the storyline)
gpcr_track <- bind_rows(go_list) %>%
  filter(Description == "G protein-coupled receptor signaling pathway") %>%
  select(top_n, Description, Count, GeneRatio, p.adjust)

print(gpcr_track)
write.csv(gpcr_track, paste0(save_dir, "GO_topN_GPCR_track.csv"),
          row.names = FALSE)

In [ ]:
# suppressMessages({ library(ggplot2); library(dplyr) })

# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# go_mac <- read.csv(paste0(save_dir, "GO_BP_macaque_specific.csv"))

# ## ---- 1. Remove terms unrelated to raphe 5-HT neurons ----
# drop_terms <- c(
#   "ossification",                                    # ossification
#   "osteoblast differentiation",                      # osteoblast differentiation
#   "epithelial cell proliferation",                   # epithelial cell proliferation
#   "positive regulation of epithelial cell proliferation",
#   "morphogenesis of a branching epithelium",         # branching epithelium morphogenesis
#   "muscle cell proliferation",                       # muscle cell proliferation
#   "response to lipopolysaccharide",                  # response to lipopolysaccharide
#   "response to molecule of bacterial origin"         # response to molecule of bacterial origin
# )

# go_show <- go_mac %>% filter(!Description %in% drop_terms)
# cat("Removed", nrow(go_mac) - nrow(go_show), "terms, remaining", nrow(go_show), " terms\n")

# ## ---- 2. Make the Top10 plot ----
# TOP_N <- 10   # ★ main figure: suggest 10

# plot_df <- go_show %>%
#   slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
#   mutate(GeneRatio = sapply(strsplit(GeneRatio, "/"), function(x)
#     as.numeric(x[1]) / as.numeric(x[2])),
#     Description = factor(Description, levels = rev(Description)))

# p <- ggplot(plot_df, aes(x = GeneRatio, y = Description)) +
#   geom_point(aes(size = Count, color = p.adjust)) +
#   scale_color_gradient(low = "red", high = "blue",
#                        limits = c(0, 0.05), name = "adj.P") +
#   scale_size_continuous(name = "Count", range = c(2, 7)) +
#   labs(title = "GO BP: macaque-specific 5-HT genes",
#        x = "GeneRatio", y = NULL) +
#   theme_bw(base_size = 10) +
#   theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 12),
#         panel.grid.major = element_blank(),
#         panel.grid.minor = element_blank(),
#         legend.position = "right")

# print(p)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_clean_Top10.png"),
#        p, width = 8, height = 5.5, dpi = 300)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_clean_Top10.pdf"),
#        p, width = 8, height = 5.5)

# ## ---- 3. Also save the full filtered table ----
# write.csv(go_show, paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
#           row.names = FALSE)

In [ ]:
# suppressMessages({ library(ggplot2); library(dplyr) })

# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# go_show <- read.csv(paste0(save_dir, "GO_BP_macaque_specific_clean.csv"))

# TOP_N <- 10

# plot_df <- go_show %>%
#   slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
#   mutate(neglogP = -log10(p.adjust),
#          label   = paste0(Description, " (", ID, ")"),
#          label   = factor(label, levels = rev(label)))

# # x-axis range: 0 to max plus 15% margin; no point gets clipped
# x_max <- max(plot_df$neglogP, na.rm = TRUE)

# p <- ggplot(plot_df, aes(x = neglogP, y = label)) +
#   geom_point(aes(size = Count, fill = neglogP),
#              shape = 21, color = "grey30", stroke = 0.3) +
#   scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
#                        midpoint = x_max / 2,
#                        name = expression(-log[10](P[adj]))) +
#   scale_size_continuous(name = "Count", range = c(3, 8)) +
#   scale_x_continuous(limits = c(0, x_max * 1.15),
#                      breaks = pretty(c(0, x_max * 1.15), n = 5),
#                      expand = c(0, 0)) +          # no default expansion; fully controlled by limits
#   labs(title = "Top GO Biological Processes",
#        x = expression(-log[10](P[adj])), y = NULL) +
#   theme_classic(base_size = 11) +
#   theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 13),
#         axis.line   = element_line(color = "black"),
#         axis.text.y = element_text(size = 9, color = "black"),
#         axis.text.x = element_text(size = 9, color = "black"),
#         axis.ticks  = element_line(color = "black"),
#         legend.position = "right",
#         legend.title = element_text(size = 9),
#         legend.text  = element_text(size = 8),
#         legend.key.size = unit(0.4, "cm"))

# print(p)

# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_neglogP_Top10_v2.png"),
#        p, width = 9, height = 6, dpi = 300)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_neglogP_Top10_v2.pdf"),
#        p, width = 9, height = 6)

In [ ]:
# suppressMessages({ library(ggplot2); library(dplyr); library(scales) })
# select <- dplyr::select

# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# go_show <- read.csv(paste0(save_dir, "GO_BP_macaque_specific_clean.csv"))

# TOP_N <- 10

# plot_df <- go_show %>%
#   slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
#   mutate(
#     GeneRatio = sapply(strsplit(GeneRatio, "/"), function(x)
#       as.numeric(x[1]) / as.numeric(x[2])),
#     neglogP = -log10(p.adjust),
#     label   = paste0(Description, " (", ID, ")"),
#     label   = factor(label, levels = rev(label)))   # y-axis still by significance: top = most significant

# print(as_tibble(select(plot_df, Description, GeneRatio, neglogP, Count)), n = Inf)

# gr_max <- max(plot_df$GeneRatio, na.rm = TRUE)

# p <- ggplot(plot_df, aes(x = GeneRatio, y = label)) +
#   geom_point(aes(size = Count, fill = neglogP),
#              shape = 21, color = "grey30", stroke = 0.3) +
#   # color = significance (-log10 Padj)
#   scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
#                        midpoint = max(plot_df$neglogP, na.rm = TRUE) / 2,
#                        name = "-log10(Padj)") +
#   scale_size_continuous(name = "Count", range = c(3, 8)) +
#   # x-axis: GeneRatio, from 0, 0.05 steps, no clipping
#   scale_x_continuous(limits = c(0, ceiling(gr_max * 20) / 20),
#                      breaks = seq(0, ceiling(gr_max * 20) / 20, by = 0.05),
#                      labels = number_format(accuracy = 0.05, trim = TRUE),
#                      expand = c(0, 0)) +
#   labs(title = "Top GO Biological Processes",
#        x = "GeneRatio", y = NULL) +
#   theme_classic(base_size = 11) +
#   theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 13),
#         axis.line   = element_line(color = "black"),
#         axis.text.y = element_text(size = 9, color = "black"),
#         axis.text.x = element_text(size = 9, color = "black"),
#         axis.ticks  = element_line(color = "black"),
#         legend.position = "right",
#         legend.title = element_text(size = 9),
#         legend.text  = element_text(size = 8),
#         legend.key.size = unit(0.4, "cm"))

# print(p)

# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_v3.png"),
#        p, width = 9, height = 6, dpi = 300)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_v3.pdf"),
#        p, width = 9, height = 6)

In [ ]:
# suppressMessages({ library(ggplot2); library(dplyr); library(scales) })
# select <- dplyr::select   # avoid AnnotationDbi conflict

# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# go_show <- read.csv(paste0(save_dir, "GO_BP_macaque_specific_clean.csv"))

# TOP_N <- 10

# plot_df <- go_show %>%
#   slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
#   mutate(
#     GeneRatio = sapply(strsplit(GeneRatio, "/"), function(x)
#       as.numeric(x[1]) / as.numeric(x[2])),
#     neglogP = -log10(p.adjust),
#     label   = paste0(Description, " (", ID, ")"),
#     # only change here: y-axis by GeneRatio descending (was rev(label), by significance)
#     label   = factor(label, levels = label[order(GeneRatio)]))

# # check values
# print(as_tibble(select(plot_df, Description, GeneRatio, neglogP, Count)), n = Inf)

# gr_max <- max(plot_df$GeneRatio, na.rm = TRUE)

# p <- ggplot(plot_df, aes(x = GeneRatio, y = label)) +
#   geom_point(aes(size = Count, fill = neglogP),
#              shape = 21, color = "grey30", stroke = 0.3) +
#   # color still a significance gradient
#   scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
#                        midpoint = max(plot_df$neglogP, na.rm = TRUE) / 2,
#                        name = "-log10(Padj)") +
#   scale_size_continuous(name = "Count", range = c(3, 8)) +
#   # GeneRatio axis: start from 0, evenly spaced integer ticks, no clipping
#   scale_x_continuous(limits = c(0, ceiling(gr_max * 20) / 20),
#                      breaks = seq(0, ceiling(gr_max * 20) / 20, by = 0.05),
#                      labels = number_format(accuracy = 0.05, trim = TRUE),
#                      expand = c(0, 0)) +
#   labs(title = "Top GO Biological Processes",
#        x = "GeneRatio", y = NULL) +
#   theme_classic(base_size = 11) +
#   theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 13),
#         axis.line   = element_line(color = "black"),
#         axis.text.y = element_text(size = 9, color = "black"),
#         axis.text.x = element_text(size = 9, color = "black"),
#         axis.ticks  = element_line(color = "black"),
#         legend.position = "right",
#         legend.title = element_text(size = 9),
#         legend.text  = element_text(size = 8),
#         legend.key.size = unit(0.4, "cm"))

# print(p)

# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_Top10.png"),
#        p, width = 9, height = 6, dpi = 300)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_Top10.pdf"),
#        p, width = 9, height = 6)

In [ ]:
# suppressMessages({ library(ggplot2); library(dplyr); library(scales) })
# select <- dplyr::select   # avoid AnnotationDbi conflict

# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# go_show <- read.csv(paste0(save_dir, "GO_BP_macaque_specific_clean.csv"))

# TOP_N <- 10

# plot_df <- go_show %>%
#   slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
#   mutate(
#     GeneRatio = sapply(strsplit(GeneRatio, "/"), function(x)
#       as.numeric(x[1]) / as.numeric(x[2])),
#     neglogP = -log10(p.adjust),
#     label   = paste0(Description, " (", ID, ")"),
#     # only change here: y-axis by GeneRatio descending (was rev(label), by significance)
#     label   = factor(label, levels = label[order(GeneRatio)]))

# # check values
# print(as_tibble(select(plot_df, Description, GeneRatio, neglogP, Count)), n = Inf)

# gr_max <- max(plot_df$GeneRatio, na.rm = TRUE)

# p <- ggplot(plot_df, aes(x = GeneRatio, y = label)) +
#   geom_point(aes(size = Count, fill = neglogP),
#              shape = 21, color = "grey30", stroke = 0.3) +
#   # color still a significance gradient
#   scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
#                        midpoint = max(plot_df$neglogP, na.rm = TRUE) / 2,
#                        name = "-log10(Padj)") +
#   scale_size_continuous(name = "Count", range = c(3, 8)) +
#   # GeneRatio axis: start from 0, evenly spaced integer ticks, no clipping
#   scale_x_continuous(limits = c(0, ceiling(gr_max * 20) / 20),
#                      breaks = seq(0, ceiling(gr_max * 20) / 20, by = 0.05),
#                      labels = number_format(accuracy = 0.05, trim = TRUE),
#                      expand = c(0, 0)) +
#   labs(title = "Top GO Biological Processes",
#        x = "GeneRatio", y = NULL) +
#   theme_classic(base_size = 11) +
#   theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 13),
#         axis.line   = element_line(color = "black"),
#         axis.text.y = element_text(size = 9, color = "black"),
#         axis.text.x = element_text(size = 9, color = "black"),
#         axis.ticks  = element_line(color = "black"),
#         legend.position = "right",
#         legend.title = element_text(size = 9),
#         legend.text  = element_text(size = 8),
#         legend.key.size = unit(0.4, "cm"))

# print(p)

# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_Top10.png"),
#        p, width = 8.63, height = 3, dpi = 300)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_Top10.pdf"),
#        p, width = 8.63, height = 3)

In [ ]:
# suppressMessages({ library(ggplot2); library(dplyr); library(scales) })
# select <- dplyr::select

# save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# go_show <- read.csv(paste0(save_dir, "GO_BP_macaque_specific_clean.csv"))

# TOP_N <- 10

# plot_df <- go_show %>%
#   slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
#   mutate(
#     GeneRatio = sapply(strsplit(GeneRatio, "/"), function(x)
#       as.numeric(x[1]) / as.numeric(x[2])),
#     neglogP = -log10(p.adjust),
#     label   = paste0(Description, " (", ID, ")"),
#     label   = factor(label, levels = label[order(GeneRatio)]))

# print(as_tibble(select(plot_df, Description, GeneRatio, neglogP, Count)), n = Inf)

# gr_max <- max(plot_df$GeneRatio, na.rm = TRUE)

# p <- ggplot(plot_df, aes(x = GeneRatio, y = label)) +
#   geom_point(aes(size = Count, fill = neglogP),
#              shape = 21, color = "grey30", stroke = 0.3) +
#   scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
#                        midpoint = max(plot_df$neglogP, na.rm = TRUE) / 2,
#                        name = "-log10(Padj)") +
#   scale_size_continuous(name = "Count", range = c(1.5, 4)) +
#   scale_x_continuous(limits = c(0, ceiling(gr_max * 20) / 20),
#                      breaks = seq(0, ceiling(gr_max * 20) / 20, by = 0.05),
#                      labels = number_format(accuracy = 0.05, trim = TRUE),
#                      expand = c(0, 0)) +
#   labs(title = "Top GO Biological Processes",
#        x = "GeneRatio", y = NULL) +
#   # Arial + 8pt: base_family = "sans" (system default maps to Arial)
#   theme_classic(base_size = 8, base_family = "sans") +
#   theme(text        = element_text(family = "sans", size = 8),
#         plot.title  = element_text(family = "sans", size = 8, face = "bold",
#                                    hjust = 0.5),
#         axis.text.y = element_text(family = "sans", size = 8, color = "black"),
#         axis.text.x = element_text(family = "sans", size = 8, color = "black"),
#         axis.title  = element_text(family = "sans", size = 8),
#         axis.line   = element_line(color = "black", linewidth = 0.3),
#         axis.ticks  = element_line(color = "black", linewidth = 0.3),
#         legend.position = "right",
#         legend.title = element_text(family = "sans", size = 8),
#         legend.text  = element_text(family = "sans", size = 7),
#         legend.key.size = unit(0.25, "cm"))

# print(p)

# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_Arial8pt.png"),
#        p, width = 8.63, height = 3, dpi = 300)
# ggsave(paste0(save_dir, "GO_BP_macaque_specific_dot_GeneRatio_Arial8pt.pdf"),
#        p, width = 8.63, height = 3)

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_1011/"

# ---- 1. Gene set and background ----
mac_genes <- gene_table %>% filter(conservation == "Macaque-specific") %>% pull(gene)
bg_genes  <- rownames(mk)          # use all genes tested by FindMarkers as background; more rigorous than the whole genome
cat("macaque-specific:", length(mac_genes), " background:", length(bg_genes), "\n")

# ---- 2. GO enrichment (BP mainly; all three ontologies can be run) ----
run_go <- function(ont) {
  enrichGO(gene          = mac_genes,
           universe      = bg_genes,
           OrgDb         = org.Hs.eg.db,
           keyType       = "SYMBOL",
           ont           = ont,
           pAdjustMethod = "BH",
           pvalueCutoff  = 0.05,
           qvalueCutoff  = 0.2,
           readable      = TRUE)
}

ego_bp <- run_go("BP")
ego_cc <- run_go("CC")
ego_mf <- run_go("MF")

# check mapping rate (losing some macaque symbols -> human Entrez is normal)
cat("BP enriched to ", nrow(as.data.frame(ego_bp)), " pathways\n")

# ---- 3. Save results table ----
write.csv(as.data.frame(ego_bp), paste0(save_dir, "GO_BP_macaque_specific.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_cc), paste0(save_dir, "GO_CC_macaque_specific.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_mf), paste0(save_dir, "GO_MF_macaque_specific.csv"), row.names = FALSE)

# ---- 4. Plot ----
# barplot
p_bar <- barplot(ego_bp, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot.pdf"), p_bar, width = 8, height = 7)

# dotplot
p_dot <- dotplot(ego_bp, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot.pdf"), p_dot, width = 8, height = 7)

# gene-pathway network (most intuitive: which genes drive the enrichment)
p_net <- cnetplot(ego_bp, showCategory = 8, circular = FALSE,
                  colorEdge = TRUE, node_label = "gene",
                  cex_label_category = 0.9, cex_label_gene = 0.7)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_cnetplot.pdf"), p_net, width = 11, height = 9)

# clustering tree (simplify redundant pathways)
ego_bp_s <- pairwise_termsim(ego_bp)
p_tree <- treeplot(ego_bp_s, showCategory = 20)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot.pdf"), p_tree, width = 9, height = 7)

print(p_dot)

In [ ]:
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_2/"
# ---- 4. Plot (remove non-raphe/contamination-related terms first) ----
# keywords of terms to remove: cilia, sensory/visual, sperm, appendage/limb, immune/inflammation
drop_kw <- paste(c(
  "cilium", "flagell", "sperm",
  "sensory perception", "visual perception", "detection of",
  "phototransduction", "appendage", "limb",
  "immune", "inflammatory", "leukocyte", "lymphocyte",
  "defense response", "complement", "cytokine"
), collapse = "|")

res <- ego_bp@result
cat("before:", nrow(res), " terms\n")
keep <- !grepl(drop_kw, res$Description, ignore.case = TRUE)
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[keep, ]
cat("after:", nrow(ego_bp_filt@result), " terms\n")
print(ego_bp_filt@result$Description)

# save the cleaned table
write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)

# dotplot (cleaned)
p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 6)
print(p_dot)

# barplot (cleaned)
p_bar <- barplot(ego_bp_filt, showCategory = 15, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_clean.pdf"),
       p_bar, width = 8, height = 6)

# tree (cleaned; see term clustering)
ego_filt_s <- pairwise_termsim(ego_bp_filt)
p_tree <- treeplot(ego_filt_s, showCategory = 15)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot_clean.pdf"),
       p_tree, width = 9, height = 6)

In [ ]:
# keep only adhesion/ECM/cell-junction/matrix-related terms; hide all others
keep_kw <- paste(c(
  "adhesion", "extracellular matrix", "extracellular structure",
  "encapsulating structure", "cell junction", "cell-cell adhesion",
  "integrin", "focal adhesion", "substrate adhesion"
), collapse = "|")

res <- ego_bp@result
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[grepl(keep_kw, res$Description, ignore.case = TRUE), ]
cat("kept:", nrow(ego_bp_filt@result), " terms\n")
print(ego_bp_filt@result$Description)

write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"), row.names = FALSE)

p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 10) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 5)
print(p_dot)

In [ ]:
# ---- 4. Plot (remove non-raphe/contamination-related terms first) ----
# keywords of terms to remove: cilia, sensory/visual, sperm, appendage/limb, immune/inflammation
drop_kw <- paste(c(
  "cilium", "flagell", "sperm",
  "sensory perception", "visual perception", "detection of",
  "phototransduction", "appendage", "limb",
  "immune", "inflammatory", "leukocyte", "lymphocyte",
  "defense response", "complement", "cytokine"
), collapse = "|")

res <- ego_bp@result
cat("before:", nrow(res), " terms\n")
keep <- !grepl(drop_kw, res$Description, ignore.case = TRUE)
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[keep, ]
cat("after:", nrow(ego_bp_filt@result), " terms\n")
print(ego_bp_filt@result$Description)

# save the cleaned table
write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)

# dotplot (cleaned)
p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 6)
print(p_dot)

# barplot (cleaned)
p_bar <- barplot(ego_bp_filt, showCategory = 15, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_clean.pdf"),
       p_bar, width = 8, height = 6)

# tree (cleaned; see term clustering)
ego_filt_s <- pairwise_termsim(ego_bp_filt)
p_tree <- treeplot(ego_filt_s, showCategory = 15)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot_clean.pdf"),
       p_tree, width = 9, height = 6)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(dplyr)
  library(ggplot2); library(clusterProfiler); library(org.Hs.eg.db)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/cross_species_2/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 0. All macaque DEGs (no top-N truncation) ----
sig <- mk$p_val_adj < 0.05
genes_mac <- rownames(mk)[sig & mk$avg_log2FC > 0.5]   # all of them, none dropped
bg_genes  <- rownames(mk)                               # background = all tested genes
cat("macaque-specific DEGs:", length(genes_mac), "\n")

# ---- optional: remove suspected contamination genes (enable after confirming with FeaturePlot) ----
# contam_genes <- c("DNAH9","CFAP157","FOXJ1","RHO","PDE6B","CNGB1",   # cilia/photoreceptor
#                   "ACTG2","TAGLN","MYH11","RGS5","KCNJ8","ABCC9")    # smooth muscle/pericyte
# genes_mac <- setdiff(genes_mac, contam_genes)
# cat("after decontamination:", length(genes_mac), "\n")

pal_fill <- scale_fill_gradientn(colors = c("red", "orange", "purple", "blue"),
                                 values = c(0, 0.33, 0.66, 1))
pal_col  <- scale_color_gradientn(colors = c("red", "orange", "purple", "blue"),
                                  values = c(0, 0.33, 0.66, 1))

## =========================================================================
## 1. GO BP (all DEGs, enrichGO)
## =========================================================================
ego <- enrichGO(gene = genes_mac, universe = bg_genes,
                OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
                ont = "BP", pAdjustMethod = "BH",
                pvalueCutoff = 0.05, qvalueCutoff = 0.2,
                readable = TRUE)

go_tab <- as.data.frame(ego) %>%
  mutate(neglogP = -log10(pvalue),
         Description = str_wrap(Description, 40)) %>%
  arrange(desc(neglogP))
write.csv(go_tab, paste0(save_dir, "GO_BP_macaque_specific_allDEG.csv"),
          row.names = FALSE)
cat("significant GO BP pathways:", nrow(go_tab), "\n")

# Top 10 barplot
top10 <- head(go_tab, 10) %>%
  mutate(Description = factor(Description, levels = rev(Description)))
p_go <- ggplot(top10, aes(neglogP, Description, fill = neglogP)) +
  geom_col() + pal_fill +
  labs(x = "-log10(P-value)", y = NULL, fill = "-log10(P-value)",
       title = "Top GO Biological Processes (macaque-specific)") +
  theme_classic(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 12))
ggsave(paste0(save_dir, "GO_BP_macaque_specific_allDEG_top10.pdf"),
       p_go, width = 7.5, height = 5)
print(p_go)

# dotplot version (20 terms)
p_dot <- dotplot(ego, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes (all DEGs)")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_allDEG_dotplot.pdf"),
       p_dot, width = 7.5, height = 8)

## =========================================================================
## 2. KEGG (all DEGs, symbol -> Entrez then enrichKEGG)
## =========================================================================
df_g <- bitr(genes_mac, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
df_b <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
cat("Entrez mapping:", nrow(df_g), "/", length(genes_mac), "\n")

ekegg <- enrichKEGG(gene = df_g$ENTREZID, universe = df_b$ENTREZID,
                    organism = "hsa", keyType = "kegg",
                    pAdjustMethod = "BH",
                    pvalueCutoff = 0.05, qvalueCutoff = 0.2)

kegg_tab <- as.data.frame(ekegg) %>%
  mutate(neglogP = -log10(pvalue)) %>%
  arrange(desc(neglogP))
write.csv(kegg_tab, paste0(save_dir, "KEGG_macaque_specific_allDEG.csv"),
          row.names = FALSE)
cat("significant KEGG pathways:", nrow(kegg_tab), "\n")

# subset neuro/synapse-related terms separately (focus on Serotonergic synapse)
neuro <- kegg_tab[grep("seroton|synap|dopamin|glutamat|calcium|neuroactive|long-term|ECM|focal adhesion",
                       kegg_tab$Description, ignore.case = TRUE), ]
write.csv(neuro, paste0(save_dir, "KEGG_macaque_specific_neuro_related.csv"),
          row.names = FALSE)
if (nrow(neuro) > 0) {
  cat("neuro/synapse/matrix-related KEGG:\n")
  print(neuro[, c("Description", "pvalue", "Count")])
}

# Top 5 dotplot
if (nrow(kegg_tab) > 0) {
  top5 <- head(kegg_tab, 5) %>%
    mutate(Description = str_to_upper(Description),
           Description = factor(Description, levels = rev(Description)))
  p_kegg <- ggplot(top5, aes(neglogP, Description, size = Count)) +
    geom_point(aes(color = neglogP)) + pal_col +
    scale_size_continuous(range = c(2, 8), name = "Gene Counts") +
    labs(x = "-log10(P-value)", y = "KEGG Terms", color = "-log10(P-value)",
         title = "Top 5 KEGG (macaque-specific)") +
    theme_classic(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 12))
  ggsave(paste0(save_dir, "KEGG_macaque_specific_allDEG_top5.pdf"),
         p_kegg, width = 6.5, height = 4.5)
  print(p_kegg)
}

cat("All done, results in", save_dir, "\n")

In [ ]:
contam_markers <- c("DNAH9","CFAP157","FOXJ1",      # cilia/ependymal
                    "RHO","PDE6B","CNGB1",          # photoreceptor
                    "ACTG2","TAGLN","MYH11","RGS5") # smooth muscle/pericyte
p_check <- FeaturePlot(classFiltSeurat, features = contam_markers,
                       reduction = "umap", ncol = 4, order = TRUE)
print(p_check)
ggsave(paste0(save_dir, "Check_contamination_featureplot.pdf"),
       p_check, width = 16, height = 8)

In [ ]:
# 1. First quantify how many cells are affected (good to know)
contam_genes_detect <- intersect(contam_markers, rownames(classFiltSeurat))
n_pos <- colSums(GetAssayData(classFiltSeurat, slot = "data")[contam_genes_detect, , drop = FALSE] > 0)
cat("cells carrying >=1 contamination marker:", sum(n_pos > 0), "/", ncol(classFiltSeurat), "\n")

# 2. Remove these genes from the DEG list (for enrichment analysis only)
contam_genes <- c(contam_markers, "KCNJ8","ABCC9",        # complete pericyte list
                  "MYL9","DES","PLN")                     # additional smooth muscle/muscle genes
genes_mac_clean <- setdiff(genes_mac, contam_genes)
cat("before:", length(genes_mac), " after:", length(genes_mac_clean), "\n")

# 3. Re-run GO / KEGG (replace genes_mac with genes_mac_clean; everything else unchanged)

In [ ]:
## =========================================================================
## data-driven removal of contamination genes based on GO terms
## =========================================================================
# 1. Pull out all suspicious terms from the previous GO results (keyword matching)
bad_kw <- "cilium|flagell|sperm motility|sensory perception|visual perception|detection of stimulus|detection of abiotic|detection of external|detection of chemical|appendage|limb|phototransduction|visual system"
bad_rows <- go_tab[grep(bad_kw, go_tab$Description, ignore.case = TRUE), ]
cat("suspicious GO terms:", nrow(bad_rows), " terms\n")

# 2. All genes hit by these terms (geneID is symbols separated by / when readable=TRUE)
bad_genes <- unique(unlist(strsplit(bad_rows$geneID, "/")))
cat("genes involved:", length(bad_genes), " genes\n")

# 3. Remove them from the DEGs
genes_mac_clean <- setdiff(genes_mac, bad_genes)
cat("before:", length(genes_mac), " → after:", length(genes_mac_clean), "\n")

# 4. Re-run GO / KEGG (replace genes_mac with genes_mac_clean; background unchanged)

In [ ]:
## =========================================================================
## 4. Re-run GO BP with genes_mac_clean
## =========================================================================
ego_clean <- enrichGO(gene = genes_mac_clean, universe = bg_genes,
                      OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
                      ont = "BP", pAdjustMethod = "BH",
                      pvalueCutoff = 0.05, qvalueCutoff = 0.2,
                      readable = TRUE)

go_clean <- as.data.frame(ego_clean) %>%
  mutate(neglogP = -log10(pvalue),
         Description = str_wrap(Description, 40)) %>%
  arrange(desc(neglogP))
write.csv(go_clean, paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)
cat("significant GO BP pathways after decontamination:", nrow(go_clean), "\n")

# Top 10 barplot
top10 <- head(go_clean, 10) %>%
  mutate(Description = factor(Description, levels = rev(Description)))
p_go <- ggplot(top10, aes(neglogP, Description, fill = neglogP)) +
  geom_col() +
  scale_fill_gradientn(colors = c("red", "orange", "purple", "blue"),
                       values = c(0, 0.33, 0.66, 1)) +
  labs(x = "-log10(P-value)", y = NULL, fill = "-log10(P-value)",
       title = "Top GO Biological Processes (macaque-specific, cleaned)") +
  theme_classic(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 12))
ggsave(paste0(save_dir, "GO_BP_macaque_specific_clean_top10.pdf"),
       p_go, width = 7.5, height = 5)
print(p_go)

# Top 20 dotplot
p_dot <- dotplot(ego_clean, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes (cleaned)")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_clean_dotplot.pdf"),
       p_dot, width = 7.5, height = 8)

## =========================================================================
## 5. Re-run KEGG with genes_mac_clean
## =========================================================================
df_g <- bitr(genes_mac_clean, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
df_b <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
cat("Entrez mapping:", nrow(df_g), "/", length(genes_mac_clean), "\n")

ekegg_clean <- enrichKEGG(gene = df_g$ENTREZID, universe = df_b$ENTREZID,
                          organism = "hsa", keyType = "kegg",
                          pAdjustMethod = "BH",
                          pvalueCutoff = 0.05, qvalueCutoff = 0.2)

kegg_clean <- as.data.frame(ekegg_clean) %>%
  mutate(neglogP = -log10(pvalue)) %>%
  arrange(desc(neglogP))
write.csv(kegg_clean, paste0(save_dir, "KEGG_macaque_specific_clean.csv"),
          row.names = FALSE)
cat("significant KEGG pathways after decontamination:", nrow(kegg_clean), "\n")

# subset neuro/synapse/matrix-related terms separately
neuro <- kegg_clean[grep("seroton|synap|dopamin|glutamat|calcium|neuroactive|long-term|ECM|focal adhesion|adherens|tight junction|gap junction",
                         kegg_clean$Description, ignore.case = TRUE), ]
write.csv(neuro, paste0(save_dir, "KEGG_macaque_specific_clean_neuro_related.csv"),
          row.names = FALSE)
if (nrow(neuro) > 0) {
  cat("neuro/synapse/matrix-related KEGG:\n")
  print(neuro[, c("Description", "pvalue", "Count")])
} else {
  cat("no neuro/synapse-related KEGG pathways hit\n")
}

# Top 5 dotplot
if (nrow(kegg_clean) > 0) {
  top5 <- head(kegg_clean, 5) %>%
    mutate(Description = str_to_upper(Description),
           Description = factor(Description, levels = rev(Description)))
  p_kegg <- ggplot(top5, aes(neglogP, Description, size = Count)) +
    geom_point(aes(color = neglogP)) +
    scale_color_gradientn(colors = c("red", "orange", "purple", "blue"),
                          values = c(0, 0.33, 0.66, 1)) +
    scale_size_continuous(range = c(2, 8), name = "Gene Counts") +
    labs(x = "-log10(P-value)", y = "KEGG Terms", color = "-log10(P-value)",
         title = "Top 5 KEGG (macaque-specific, cleaned)") +
    theme_classic(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 12))
  ggsave(paste0(save_dir, "KEGG_macaque_specific_clean_top5.pdf"),
         p_kegg, width = 6.5, height = 4.5)
  print(p_kegg)
}

## =========================================================================
## 6. Before/after comparison table (for the main text/supplementary material)
## =========================================================================
compare_tab <- data.frame(
  category = names(n_stat)[2],                    # Macaque-specific
  n_before = length(genes_mac),
  n_after  = length(genes_mac_clean),
  n_removed = length(genes_mac) - length(genes_mac_clean)
)
write.csv(compare_tab, paste0(save_dir, "Table_DEG_clean_summary.csv"),
          row.names = FALSE)
cat("All done, results in", save_dir, "\n")

In [ ]:
ekegg_relax <- enrichKEGG(gene = df_g$ENTREZID, universe = df_b$ENTREZID,
                          organism = "hsa", keyType = "kegg",
                          pAdjustMethod = "BH",
                          pvalueCutoff = 0.2, qvalueCutoff = 0.5)  # relaxed
tab <- as.data.frame(ekegg_relax)
# manually inspect ECM/synapse-related terms; can report even if p_adj is between 0.05-0.2
neuro <- tab[grep("seroton|synap|ECM|focal adhesion|calcium|neuroactive",
                  tab$Description, ignore.case = TRUE), ]
print(neuro[, c("Description", "pvalue", "p.adjust", "Count")])

In [ ]:
# use the full ranking before cleaning (mk_clean or mk); just remove contamination genes from the ranking
geneList <- mk$avg_log2FC
names(geneList) <- mk$gene
geneList <- geneList[!names(geneList) %in% bad_genes]   # bad_genes: suspicious genes from the previous step
geneList <- sort(geneList, decreasing = TRUE)

# symbol -> Entrez
df_all <- bitr(names(geneList), fromType = "SYMBOL",
               toType = "ENTREZID", OrgDb = org.Hs.eg.db)
geneList <- geneList[df_all$SYMBOL]
names(geneList) <- df_all$ENTREZID
geneList <- sort(geneList, decreasing = TRUE)

gse <- gseKEGG(geneList = geneList, organism = "hsa",
               keyType = "kegg", pvalueCutoff = 0.05,
               pAdjustMethod = "BH", verbose = FALSE)
gse_tab <- as.data.frame(gse)
write.csv(gse_tab, paste0(save_dir, "GSEA_KEGG_macaque_vs_mouse_clean.csv"),
          row.names = FALSE)

# look at NES>0 (up in macaque) synapse/ECM pathways
neuro_gse <- gse_tab[grep("seroton|synap|ECM|focal adhesion|calcium|neuroactive|long-term",
                          gse_tab$Description, ignore.case = TRUE) & gse_tab$NES > 0, ]
print(neuro_gse[, c("Description", "NES", "p.adjust")])

In [ ]:
ribo <- grep("^RPL|^RPS|^MRPL|^MRPS", rownames(mk), value = TRUE)
ribo_entrez <- bitr(ribo, fromType = "SYMBOL", toType = "ENTREZID",
                    OrgDb = org.Hs.eg.db)$ENTREZID
test_kegg <- enrichKEGG(gene = ribo_entrez, organism = "hsa",
                        pvalueCutoff = 0.05)
print(head(as.data.frame(test_kegg)))

In [ ]:
# metaneighbor

In [ ]:
formals(MetaNeighborUS)

In [ ]:
suppressMessages(library(MetaNeighbor))
suppressMessages(library(SummarizedExperiment))

## ---- 1. Prepare input ----
expr_mat <- GetAssayData(classFiltSeurat[["SCT"]], slot = "data")
use_genes <- intersect(VariableFeatures(classFiltSeurat), rownames(expr_mat))
expr_mat <- as.matrix(expr_mat[use_genes, ])

keep_cells <- colnames(classFiltSeurat)[classFiltSeurat$subtype_recl != "Unassigned"]
meta_nn <- data.frame(
  cell_id    = keep_cells,
  study_id   = as.character(classFiltSeurat$species[keep_cells]),
  cell_label = as.character(classFiltSeurat$subtype_recl[keep_cells]),
  row.names  = keep_cells,
  stringsAsFactors = FALSE
)

# key change: wrap into a SummarizedExperiment
se_nn <- SummarizedExperiment(
  assays  = list(counts = expr_mat[, keep_cells]),
  colData = meta_nn[keep_cells, ]
)

## ---- 2. Select gene set ----
var_genes <- variableGenes(dat = se_nn, exp_labels = se_nn$study_id)
message("genes used by MetaNeighbor: ", length(var_genes))

## ---- 3. Run MetaNeighborUS (genesets must be a list; pass se_nn to dat) ----
set.seed(42)
celltype_NV <- MetaNeighborUS(
  var_genes  = var_genes,          # pass the gene vector directly; no list() needed
  dat        = se_nn,
  study_id   = se_nn$study_id,
  cell_type  = se_nn$cell_label
  # fast_version = TRUE speeds things up (enable if you have enough memory)
)

In [ ]:
dim(celltype_NV)
head(rownames(celltype_NV))
head(colnames(celltype_NV))

In [ ]:
## ---- 3. Orient: rows = macaque, columns = mouse ----
parse_name <- function(x) {
  parts <- do.call(rbind, strsplit(x, "\\|"))
  data.frame(study = parts[, 1], label = parts[, 2],
             stringsAsFactors = FALSE)
}

rn <- parse_name(rownames(celltype_NV))
cn <- parse_name(colnames(celltype_NV))

mac_rows <- rownames(celltype_NV)[rn$study == "macaque"]
mus_cols <- colnames(celltype_NV)[cn$study == "mouse"]

NV <- celltype_NV[mac_rows, mus_cols, drop = FALSE]
rownames(NV) <- sub("^macaque\\|", "", mac_rows)
colnames(NV) <- sub("^mouse\\|",   "", mus_cols)

saveRDS(NV, paste0(out_dir, "celltype_NV_MetaNeighbor.rds"))

## ---- 4. Heatmap + best-match table (your original logic here stays as is) ----
best_match_nn <- data.frame(
  macaque_subtype = rownames(NV),
  best_mouse      = colnames(NV)[apply(NV, 1, which.max)],
  AUROC           = round(apply(NV, 1, max), 3)
)
print(best_match_nn)
write.csv(best_match_nn, paste0(out_dir, "MetaNeighbor_best_match.csv"),
          row.names = FALSE)

In [ ]:
# ---- 4. Heatmap (rows macaque / columns mouse) ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

ht <- Heatmap(
  NV,
  name = "AUROC",
  col  = colorRamp2(c(0.4, 0.5, 0.75, 1),
                    c("#3778AE", "#F7F7F7", "#F4A582", "#B2182B")),
  cluster_rows = TRUE,  cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  row_names_side = "left",
  row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
  width = unit(7, "cm"), height = unit(8, "cm"),
  column_title = "mouse", row_title = "macaque"
)

png(paste0(out_dir, "MetaNeighbor_macaqueVSmouse.png"),
    width = 9, height = 8.5, units = "in", res = 300)
draw(ht, heatmap_legend_side = "right")
dev.off()

# best-match table per row (AUROC interpretation: >0.75 = highly conserved)
best_match_nn <- data.frame(
  macaque_subtype = sub("^macaque\\|", "", rownames(NV)),
  best_mouse      = sub("^mouse\\|", "", colnames(NV)[apply(NV, 1, which.max)]),
  AUROC           = round(apply(NV, 1, max), 3)
)
print(best_match_nn)
write.csv(best_match_nn, paste0(out_dir, "MetaNeighbor_best_match.csv"),
          row.names = FALSE)

In [ ]:
ht <- Heatmap(
  NV,
  name = "AUROC",
  col  = colorRamp2(c(0.4, 0.5, 0.75, 1),
                    c("#3778AE", "#F7F7F7", "#F4A582", "#B2182B")),
  cluster_rows = TRUE,  cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  row_names_side = "left",
  row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
  width = unit(7, "cm"), height = unit(8, "cm"),
  column_title = "mouse", row_title = "macaque",

  # write the values into the cells
  cell_fun = function(j, i, x, y, width, height, fill) {
    v <- NV[i, j]
    grid.text(sprintf("%.2f", v), x, y,
              gp = gpar(fontsize = 6.5,
                        # white text on dark background, black text on light background, for readability
                        col = ifelse(v > 0.8, "white", "black")))
  }
)

png(paste0(out_dir, "MetaNeighbor_macaqueVSmouse.png"),
    width = 9, height = 8.5, units = "in", res = 300)
draw(ht, heatmap_legend_side = "right")
dev.off()

In [ ]:
# Sankey diagram

In [ ]:
library(reshape2)
library(ggalluvial)
library(ggplot2)

# full pairing, no filtering; rows macaque, columns mouse
mac_types <- rownames(NV)   # macaque|mac5HT_x
mus_types <- colnames(NV)   # mouse|5-HT_x

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "nv")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = nv)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque ↔ mouse 5-HT subtype mapping (MetaNeighbor)")

print(p_sankey)
ggsave(paste0(out_dir, "sankey_MetaNeighbor_all.pdf"),
       p_sankey, width = 14, height = 7)

# strong correspondences only (AUROC > 0.7), cleaner plot
df_strong <- subset(df, nv > 0.7)
if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = nv)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque ↔ mouse (MetaNeighbor, AUROC > 0.7)")
  print(p_sankey_strong)
  ggsave(paste0(out_dir, "sankey_MetaNeighbor_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
}

In [ ]:
# greater than 0.6

In [ ]:
library(reshape2)
library(ggalluvial)
library(ggplot2)

# full pairing, no filtering; rows macaque, columns mouse
mac_types <- rownames(NV)   # macaque|mac5HT_x
mus_types <- colnames(NV)   # mouse|5-HT_x

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "nv")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = nv)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque ↔ mouse 5-HT subtype mapping (MetaNeighbor)")

print(p_sankey)
ggsave(paste0(out_dir, "sankey_MetaNeighbor_all.pdf"),
       p_sankey, width = 14, height = 7)

# strong correspondences only (AUROC > 0.6)
df_strong <- subset(df, nv > 0.6)          # changed 0.7 to 0.6
if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = nv)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque vs mouse (MetaNeighbor, AUROC > 0.6)")   # update title accordingly
  print(p_sankey_strong)
  ggsave(paste0(out_dir, "sankey_MetaNeighbor_strong06.pdf"),  # distinct file name
         p_sankey_strong, width = 12, height = 7)
}

In [ ]:
# DEG

In [ ]:
suppressMessages(library(tidyverse))
library(ggrepel)
library(Seurat)

# classFiltSeurat <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/crossSpecies_final_5layers.rds")

## ===== 1. Differential expression: macaque vs mouse (all 5-HT cells) =====
# RNA has 5 layers; normalize after merging
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])

DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, normalization.method = "LogNormalize",
                                 scale.factor = 10000, verbose = FALSE)

Idents(classFiltSeurat) <- "species"

deg <- FindMarkers(classFiltSeurat,
                   ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", slot = "data",
                   logfc.threshold = 0.1, min.pct = 0.1,
                   test.use = "wilcox")
saveRDS(deg, "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/DEG_macaque_vs_mouse.rds")

## ===== 2. Tidy + volcano plot =====
df <- deg %>%
  mutate(gene = rownames(deg),
         sig = case_when(p_val_adj < 0.05 & avg_log2FC >  1 ~ "up",
                         p_val_adj < 0.05 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

# label the 5 most significant genes on each side (named labels on the plot)
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% slice_min(p_val_adj, n = 5),
  df %>% filter(sig == "down") %>% slice_min(p_val_adj, n = 5)
)

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, -log10(p_val_adj), color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50") +
  geom_text_repel(data = lab, aes(label = gene), size = 3.5,
                  color = "black", max.overlaps = 30, segment.size = 0.3) +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13)

print(p)
ggsave("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/volcano_macaque_vs_mouse1.pdf",
       p, width = 7, height = 6)

In [ ]:
library(ggplot2)
library(dplyr)
library(ggrepel)

df <- deg %>%
  mutate(gene  = rownames(deg),
         y_raw = -log10(p_val_adj),
         ## y <= 50 counts as ns regardless of FC; up/down only for y > 50
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## label genes: back to top5 per side (chosen among significant genes with y > 50)
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(5),
  df %>% filter(sig == "down") %>% pick_spread(5)
)

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  ## horizontal dashed line: moved from -log10(0.05) to y = 50
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    size = 3.5, color = "black", fontface = "italic",
    box.padding = 0.8,
    point.padding = 0.5,
    force = 5, force_pull = 0.2,
    direction = "both",
    max.overlaps = Inf,
    min.segment.length = 0,
    seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13) +
  theme(plot.margin = margin(20, 40, 20, 20))

print(p)
ggsave("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/volcano_macaque_vs_mouse-50.pdf",
       p, width = 8, height = 7)

In [ ]:
# violin plots of a gene's expression in the two species

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)

# classFiltSeurat <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/crossSpecies_final_5layers.rds")
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 1. Specify genes =====
top_genes <- c("PENK", "TAC1", "NXPH2", "RIMS2", "ROBO2","CAPS2","RUFY3"
)

## check whether these three genes are all in the object (cross-species data may have gene-name matching issues)
missing_genes <- setdiff(top_genes, rownames(classFiltSeurat))
if (length(missing_genes) > 0) {
  cat("WARNING: these genes are not in the object:", paste(missing_genes, collapse = ", "), "\n")
}

cat("Genes:", paste(top_genes, collapse = ", "), "\n")

## ===== 2. Extract once + convert to long format =====
expr <- FetchData(classFiltSeurat, vars = c("species", top_genes))
expr_long <- expr %>%
  pivot_longer(cols = -species, names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = top_genes),   # ordered as specified
         species = factor(species, levels = c("macaque", "mouse")))

## ===== 3. Plot =====
p <- ggplot(expr_long, aes(x = species, y = Expression, fill = species)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.size = 0.15, alpha = 0.7, color = "grey30") +
  facet_wrap(~ gene, nrow = 1, scales = "free_y") +
  scale_fill_manual(values = c("macaque" = "#EB6368", "mouse" = "#82ADD0")) +
  labs(title = "PENK / TAC1 / NXPH2: macaque vs mouse",
       x = NULL, y = "Normalized Expression") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1))

print(p)
ggsave("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/VlnPlot_PENK_TAC1_NXPH2_macaque_vs_mouse.png",
       p, width = 10, height = 5, dpi = 300)

In [ ]:
# 1. See which rds files are in the target directory (mind case, spaces, extensions)
list.files("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/step5_data_save_907/",
           pattern = "\\.rds$")

# 2. If the above returns an empty character vector, look one level up
list.files("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/",
           pattern = "crossSpecies.*\\.rds$", recursive = TRUE)

# 3. Fallback: search the whole disk (in case the mount path has changed)
list.files("/mnt/90-connectome", pattern = "crossSpecies_CCA_dims50_mac5HT\\.rds",
           recursive = TRUE, full.names = TRUE)

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)

classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 1. Specify genes =====
top_genes <- c("PENK", "TAC1", "NXPH2", "RIMS2", "ROBO2", "CAPS2", "RUFY3")

missing_genes <- setdiff(top_genes, rownames(classFiltSeurat))
if (length(missing_genes) > 0) {
  cat("WARNING: these genes are not in the object:", paste(missing_genes, collapse = ", "), "\n")
}

## ===== 2. Extract once + convert to long format (also fetch donor and cell type columns) =====
expr <- FetchData(classFiltSeurat,
                  vars = c("species", "donor", "subtype_recl", top_genes))

# check the actual values of donor and cell type (also check for NAs)
table(expr$donor, useNA = "ifany")
table(expr$subtype_recl, useNA = "ifany")

expr_long <- expr %>%
  pivot_longer(cols = all_of(top_genes),
               names_to = "gene", values_to = "Expression") %>%
  mutate(gene   = factor(gene, levels = top_genes),
         donor  = factor(donor),
         celltype = factor(subtype_recl))

## ===== 3. Plot: x = cell type, fill = donor, facet = gene =====
p <- ggplot(expr_long, aes(x = celltype, y = Expression, fill = donor)) +
  geom_violin(scale = "width", trim = TRUE, position = position_dodge(width = 0.9)) +
  geom_boxplot(width = 0.12, outlier.size = 0, alpha = 0.6, color = "grey30",
               position = position_dodge(width = 0.9)) +
  facet_wrap(~ gene, nrow = 2, scales = "free_y") +
  labs(title = "Expression across cell types and donors",
       x = NULL, y = "Normalized Expression", fill = "Donor") +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold", size = 11),
        legend.position = "right")

print(p)
ggsave("/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/VlnPlot_topGenes_byDonor_celltype.png",
       p, width = 16, height = 9, dpi = 300)

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
  library(harmony)
})

## =========================================================================
## Adjustable parameters
## =========================================================================
PAR <- list(
  batch_downsample = TRUE,   # downsample each batch to the smallest batch cell count
  variable_genes   = 5000,
  npcs             = 100,
  cca_dims         = 1:30,
  nb_dims          = 1:30,
  resolution       = 0.1,
  min_cells        = 100
)

set.seed(42)
out_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/"

## =========================================================================
## 1. Load data + clean
## =========================================================================
macaque <- readRDS(paste0(out_dir, "macaque_5HT_integrated.rds"))
table(macaque$renew_type)

DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])

macaque$donor <- as.character(macaque$donor)
macaque$donor[is.na(macaque$donor)] <- as.character(macaque$Individual[is.na(macaque$donor)])
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## =========================================================================
## 2. Ortholog conversion
## =========================================================================
test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)

mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)

mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## =========================================================================
## 3. Merge small donors + downsample each batch (stratified) to the smallest batch
## =========================================================================
own_donors <- c("Mq246", "Mq277")

macaqueHomoSeurat$batch <- ifelse(
  macaqueHomoSeurat$donor %in% own_donors,
  macaqueHomoSeurat$donor,
  "NHP_published"
)
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

cat("--- cells per batch after stratification (before downsampling) ---\n")
print(table(macaqueHomoSeurat$batch))
print(table(mouseHomoSeurat$batch))

if (PAR$batch_downsample) {
  # smallest batch cell count = downsampling target (currently mouse2 = 913)
  target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                            table(mouseHomoSeurat$batch)))
  message("Downsample each batch to ", target_per_batch, " cells (stratified by subtype within batch)")
  
  # within a single object: stratified sampling per batch independently
  downsample_by_batch <- function(obj, target_per_batch) {
    ct <- obj$clusterName
    bt <- obj$batch
    cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
      n_target <- min(length(idx), target_per_batch)
      tab <- table(ct[idx])
      n_per <- round(tab / sum(tab) * n_target)
      n_per <- pmin(n_per, tab)
      n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] +
        (n_target - sum(n_per))
      unlist(lapply(names(n_per), function(x) {
        cells_x <- colnames(obj)[idx][ct[idx] == x]
        sample(cells_x, min(n_per[x], length(cells_x)))
      }))
    }))
    sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
  }
  
  macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                              cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
  mouseHomoSeurat   <- subset(mouseHomoSeurat,
                              cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
  
  cat("--- cells per batch after downsampling ---\n")
  print(table(macaqueHomoSeurat$batch))
  print(table(mouseHomoSeurat$batch))
}

## =========================================================================
## 4. split + merge (expect 5 layers, 913 cells each)
## =========================================================================
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],
                                    f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]],
                                    f = macaqueHomoSeurat$batch)

classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)

cat("--- cells per layer after merging ---\n")
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

layer_cells <- sapply(Layers(classFiltSeurat[["RNA"]]),
                      function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l)))
if (any(layer_cells < PAR$min_cells)) {
  message("small layers found, applying filter...")
  keep_layers <- names(layer_cells)[layer_cells >= PAR$min_cells]
  cells_keep <- unlist(lapply(keep_layers, function(l)
    colnames(LayerData(classFiltSeurat[["RNA"]], layer = l))))
  classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
  classFiltSeurat <- subset(classFiltSeurat, cells = cells_keep)
  classFiltSeurat[["RNA"]] <- split(classFiltSeurat[["RNA"]],
                                    f = classFiltSeurat$batch)
}

## =========================================================================
## 5. SCT + PCA
## =========================================================================
classFiltSeurat <- SCTransform(
  classFiltSeurat, assay = "RNA",
  ncells = ncol(classFiltSeurat[["RNA"]]),
  variable.features.n = PAR$variable_genes,
  method = "glmGamPoi"
)
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
print(ElbowPlot(classFiltSeurat, ndims = PAR$npcs))

## =========================================================================
## 6. CCA integration
## =========================================================================
classFiltSeurat <- IntegrateLayers(
  object = classFiltSeurat,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  normalization.method = "SCT",
  dims = PAR$cca_dims,
  verbose = TRUE
)

## =========================================================================
## 7. Clustering + UMAP
## =========================================================================
classFiltSeurat <- FindNeighbors(classFiltSeurat,
                                 reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat,
                                resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat,
                           reduction = "integrated.cca",
                           dims = PAR$nb_dims,
                           reduction.name = "umap")

## =========================================================================
## 8. Visualization
## =========================================================================
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (CCA, batch-balanced)")
print(p1)
ggsave(paste0(out_dir, "UMAP_CCA_species_batchbal.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(out_dir, "UMAP_CCA_batch_batchbal.pdf"), p2, width = 6.5, height = 5)

classFiltSeurat$subtype <- sub("^(macaque|mouse)_", "",
                               classFiltSeurat$clusterName)

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + NoLegend() + ggtitle("subtypes split by species")
print(p3)
ggsave(paste0(out_dir, "UMAP_CCA_subtype_split_batchbal.pdf"), p3, width = 12, height = 6)

# ## =========================================================================
# ## 9. Save
# ## =========================================================================
# saveRDS(classFiltSeurat, paste0(out_dir, "crossSpecies_CCA_5layers_batch913.rds"))
# writeLines(capture.output(sessionInfo()),
#            paste0(out_dir, "sessionInfo_CCA_batch913.txt"))
# message("done, ", ncol(classFiltSeurat), " cells in total (5 batches x 913)")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
  library(harmony)
})

## =========================================================================
## Adjustable parameters (PC = 50 for all; batches downsampled to 913)
## =========================================================================
PAR <- list(
  batch_downsample = TRUE,
  variable_genes   = 5000,
  npcs             = 100,
  cca_dims         = 1:50,
  nb_dims          = 1:50,
  resolution       = 0.1,
  min_cells        = 100
)

set.seed(42)
out_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data submission_data_code/"

## =========================================================================
## 1. Load data + clean
## =========================================================================
macaque <- readRDS(paste0(out_dir, "macaque_5HT_integrated.rds"))
table(macaque$renew_type)

DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])

macaque$donor <- as.character(macaque$donor)
macaque$donor[is.na(macaque$donor)] <- as.character(macaque$Individual[is.na(macaque$donor)])
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("/home/lixq/project/from_zry/data-DR1205/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## =========================================================================
## 2. Ortholog conversion
## =========================================================================
test <- read.table('/home/lixq/project/from_zry/data-DR1205/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)

mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)

mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## =========================================================================
## 3. Merge small donors + downsample each batch (stratified) to the smallest batch
## =========================================================================
own_donors <- c("Mq246", "Mq277")

macaqueHomoSeurat$batch <- ifelse(
  macaqueHomoSeurat$donor %in% own_donors,
  macaqueHomoSeurat$donor,
  "NHP_published"
)
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

cat("--- cells per batch after stratification (before downsampling) ---\n")
print(table(macaqueHomoSeurat$batch))
print(table(mouseHomoSeurat$batch))

if (PAR$batch_downsample) {
  target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                            table(mouseHomoSeurat$batch)))
  message("Downsample each batch to ", target_per_batch, " cells (stratified by subtype within batch)")
  
  downsample_by_batch <- function(obj, target_per_batch) {
    ct <- obj$clusterName
    bt <- obj$batch
    cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
      n_target <- min(length(idx), target_per_batch)
      tab <- table(ct[idx])
      n_per <- round(tab / sum(tab) * n_target)
      n_per <- pmin(n_per, tab)
      n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] +
        (n_target - sum(n_per))
      unlist(lapply(names(n_per), function(x) {
        cells_x <- colnames(obj)[idx][ct[idx] == x]
        sample(cells_x, min(n_per[x], length(cells_x)))
      }))
    }))
    sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
  }
  
  macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                              cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
  mouseHomoSeurat   <- subset(mouseHomoSeurat,
                              cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
  
  cat("--- cells per batch after downsampling ---\n")
  print(table(macaqueHomoSeurat$batch))
  print(table(mouseHomoSeurat$batch))
}

## =========================================================================
## 4. split + merge (expect 5 layers, 913 cells each)
## =========================================================================
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],
                                    f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]],
                                    f = macaqueHomoSeurat$batch)

classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)

cat("--- cells per layer after merging ---\n")
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

layer_cells <- sapply(Layers(classFiltSeurat[["RNA"]]),
                      function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l)))
if (any(layer_cells < PAR$min_cells)) {
  message("small layers found, applying filter...")
  keep_layers <- names(layer_cells)[layer_cells >= PAR$min_cells]
  cells_keep <- unlist(lapply(keep_layers, function(l)
    colnames(LayerData(classFiltSeurat[["RNA"]], layer = l))))
  classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
  classFiltSeurat <- subset(classFiltSeurat, cells = cells_keep)
  classFiltSeurat[["RNA"]] <- split(classFiltSeurat[["RNA"]],
                                    f = classFiltSeurat$batch)
}

## =========================================================================
## 5. SCT + PCA
## =========================================================================
classFiltSeurat <- SCTransform(
  classFiltSeurat, assay = "RNA",
  ncells = ncol(classFiltSeurat[["RNA"]]),
  variable.features.n = PAR$variable_genes,
  method = "glmGamPoi"
)
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
print(ElbowPlot(classFiltSeurat, ndims = PAR$npcs))

## =========================================================================
## 6. CCA integration (dims = 1:50)
## =========================================================================
classFiltSeurat <- IntegrateLayers(
  object = classFiltSeurat,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  normalization.method = "SCT",
  dims = PAR$cca_dims,
  verbose = TRUE
)

## =========================================================================
## 7. Clustering + UMAP (dims = 1:50)
## =========================================================================
classFiltSeurat <- FindNeighbors(classFiltSeurat,
                                 reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat,
                                resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat,
                           reduction = "integrated.cca",
                           dims = PAR$nb_dims,
                           reduction.name = "umap")

## =========================================================================
## 8. Visualization (macaque uses mac_subtype from the merged reclustering)
## =========================================================================
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (CCA dims50, batch-balanced)")
print(p1)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# ---- macaque: load merged reclustering result, map mac_subtype ----
mac_re <- readRDS(paste0(out_dir, "macaque_5HT_merged_reclustered.rds"))
mac_labels <- setNames(as.character(mac_re$mac_subtype), colnames(mac_re))

mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <-
  mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "",
      as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

# mapping QC (must check the output)
n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("cells missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(out_dir, "subtype_recl_vs_species.csv"))

# split plot: macaque reclustered subtypes vs mouse original labels
p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + NoLegend() +
  ggtitle("subtypes split (macaque = merged-reclustered)")
print(p3)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_subtype_split_recl.pdf"),
       p3, width = 12, height = 6)

# merged version
p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + NoLegend()
print(p4)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_subtype_all_recl.pdf"),
       p4, width = 7, height = 6)

## =========================================================================
## 9. Overlap heatmap (clusterName vs seurat_clusters)
## =========================================================================
suppressMessages({
  library(ComplexHeatmap)
  library(circlize)
  library(grid)
})

compare_cl <- function(cl, ref.cl, plot.title = NA, plot.silent = TRUE,
                       heat.colors = colorRampPalette(c("white", "grey70", "black"))(100),
                       row.cl.num = min(length(unique(cl)), length(unique(ref.cl)))) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  conf2 <- reorder_matrix(conf1)
  cl.prop.cocl <- apply(conf1, 2, function(x) {
    grid1 <- expand.grid(x, x)
    min.prop <- apply(grid1, 1, min)
  })
  cl.prop.cocl.total <- apply(cl.prop.cocl, 1, sum)
  cl.prop.cocl.m <- matrix(cl.prop.cocl.total, nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  ph1 <- pheatmap::pheatmap(conf2, cutree_rows = row.cl.num,
                            clustering_method = "ward.D2",
                            color = heat.colors, fontsize = 6,
                            main = plot.title, silent = plot.silent)
  return(list(conf = conf2, cocl = cl.prop.cocl.m, ph = ph1))
}

reorder_matrix <- function(matrix1, by.rows = TRUE) {
  if (by.rows == TRUE) {
    conf.order <- order(apply(matrix1, 1, which.max))
    matrix1.reordered <- matrix1[conf.order, ]
  } else {
    conf.order <- order(apply(matrix1, 2, which.max))
    matrix1.reordered <- matrix1[, conf.order]
  }
}

cl.conf <- compare_cl(classFiltSeurat$clusterName,
                      classFiltSeurat$seurat_clusters)
cocl <- cl.conf$cocl

cocl.subset <- cocl[grepl("macaque", rownames(cocl)),
                    grepl("mouse",   rownames(cocl))]

heat.colors <- colorRampPalette(c("white", "grey70", "black"))(100)

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")

left_anno <- rowAnnotation(
  "Max Overlap\nscore" = anno_barplot(
    row_max,
    gp = gpar(fill = bar_colors, col = NA),
    bar_width = 0.9,
    border = FALSE,
    axis_param = list(direction = "reverse")
  ),
  width = unit(3, "cm")
)

bar_legend <- Legend(
  title = "Type",
  at = c("macaque-specific", "macaque/mouse-conserved"),
  legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
  border = "black",
  labels_gp = gpar(fontsize = 10)
)

ht <- Heatmap(
  cocl.subset,
  name = "Overlap score",
  col = heat.colors,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  left_annotation = left_anno,
  row_names_side = "right",
  row_names_gp = gpar(fontsize = 10),
  column_names_gp = gpar(fontsize = 10),
  width = unit(5, "cm"),
  height = unit(7, "cm"),
)

png(paste0(out_dir, "overlap_cca_dims50_batch913.png"),
    width = 8, height = 7, units = "in", res = 300)

draw(ht, annotation_legend_side = "bottom",
     heatmap_legend_list = list(bar_legend))

decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n)) {
    grid.text(label = sprintf("%.2f", row_max[j]),
              x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5) / n, "npc"),
              just = "right",
              gp = gpar(fontsize = 8))
  }
})

decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0, 1), "npc"),
             gp = gpar(lty = 2, col = "black"))
})

dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_batch913.csv"))
cat("Overlap heatmap and value table saved\n")

# ## =========================================================================
# ## 10. Save object
# ## =========================================================================
# saveRDS(classFiltSeurat,
#         paste0(out_dir, "crossSpecies_CCA_dims50_batch913.rds"))
# writeLines(capture.output(sessionInfo()),
#            paste0(out_dir, "sessionInfo_CCA_dims50_batch913.txt"))
# message("done, ", ncol(classFiltSeurat), " cells in total (5 batches x 913)")